# Cross-dataset OD/OC segmenter validation

**Public role:** Frozen segmenter-selection and cross-dataset validation notebook.

**Relation to the manuscript:** Supports the selected ResidualUNetLite proxy and reported external Dice evidence. Methodological content is preserved.

**Frozen methodology:** The public curation changes only documentation, path abstraction, notebook metadata, and stored outputs. The experimental method and parameters are unchanged.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `05_train_external_od_oc_segmentors_cross_dataset_validation.ipynb`


In [ ]:
# Cell 1: Load paths and audit standardized dataset availability

from pathlib import Path
import json
import tarfile
import shutil
import os

import cv2
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing config: {CONFIG_PATH}")

if not PATH_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing path registry: {PATH_REGISTRY_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

path_registry = pd.read_csv(PATH_REGISTRY_PATH)
PATHS = {
    row["path_key"]: Path(row["path"])
    for _, row in path_registry.iterrows()
}

STANDARD_SIZE = 512

ARCHIVE_DIR = PATHS["external_datasets"] / "standardized_segmentation_archives"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

standardized_sources = {
    "REFUGE": {
        "local_root": Path("/content/refuge_standardized_512"),
        "archive_path": ARCHIVE_DIR / "refuge_standardized_512.tar",
        "standardized_manifest": PATHS["external_datasets"] / "standardized_segmentation" / "REFUGE" / "refuge_standardized_segmentation_manifest.csv",
        "raw_manifest": PATHS["project_data"] / "external_datasets" / "REFUGE" / "refuge_od_oc_manifest.csv",
        "expected_samples": 1200,
    },
    "PAPILA": {
        "local_root": Path("/content/papila_standardized_512"),
        "archive_path": ARCHIVE_DIR / "papila_standardized_512.tar",
        "standardized_manifest": PATHS["external_datasets"] / "standardized_segmentation" / "PAPILA" / "papila_standardized_segmentation_manifest.csv",
        "raw_manifest": PATHS["project_data"] / "external_datasets" / "PAPILA" / "papila_od_oc_manifest.csv",
        "expected_samples": 488,
    },
    "RIGA": {
        "local_root": Path("/content/riga_standardized_512"),
        "archive_path": ARCHIVE_DIR / "riga_standardized_512.tar",
        "standardized_manifest": PATHS["external_datasets"] / "standardized_segmentation" / "RIGA" / "riga_standardized_segmentation_manifest.csv",
        "raw_manifest": PATHS["project_data"] / "external_datasets" / "RIGA" / "riga_od_oc_manifest.csv",
        "expected_samples": 750,
    },
}

availability_rows = []

for dataset, spec in standardized_sources.items():
    image_dir = spec["local_root"] / "images_512"
    mask_dir = spec["local_root"] / "masks_512"

    local_images = len(list(image_dir.glob("*.jpg"))) if image_dir.exists() else 0
    local_masks = len(list(mask_dir.glob("*.png"))) if mask_dir.exists() else 0

    local_ready = (
        local_images == spec["expected_samples"] and
        local_masks == spec["expected_samples"]
    )

    archive_exists = spec["archive_path"].exists()
    raw_manifest_exists = spec["raw_manifest"].exists()
    standardized_manifest_exists = spec["standardized_manifest"].exists()

    if local_ready:
        action = "use_local"
    elif archive_exists:
        action = "restore_from_archive"
    elif raw_manifest_exists:
        action = "rebuild_from_raw_manifest"
    else:
        action = "missing_required_inputs"

    availability_rows.append(
        {
            "dataset": dataset,
            "local_root": str(spec["local_root"]),
            "local_images": local_images,
            "local_masks": local_masks,
            "expected_samples": spec["expected_samples"],
            "local_ready": local_ready,
            "archive_exists": archive_exists,
            "raw_manifest_exists": raw_manifest_exists,
            "standardized_manifest_exists": standardized_manifest_exists,
            "action": action,
        }
    )

standardized_availability = pd.DataFrame(availability_rows)

availability_csv = PATHS["results_04_od_segmentor_external_validation"] / "notebook05_standardized_availability_audit.csv"
standardized_availability.to_csv(availability_csv, index=False)

print(
    standardized_availability[
        [
            "dataset",
            "local_images",
            "local_masks",
            "archive_exists",
            "raw_manifest_exists",
            "action",
        ]
    ].to_string(index=False)
)
print(f"Availability audit: {availability_csv}")

missing_inputs = standardized_availability.loc[
    standardized_availability["action"] == "missing_required_inputs",
    "dataset",
].tolist()

if missing_inputs:
    raise RuntimeError(f"Missing required inputs for datasets: {missing_inputs}")

print("Status: PASS")

In [ ]:
# Cell 2: Define standardized dataset restore, rebuild, and archive functions

def reset_directory(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def create_tar_archive(source_dir, archive_path):
    source_dir = Path(source_dir)
    archive_path = Path(archive_path)
    archive_path.parent.mkdir(parents=True, exist_ok=True)

    local_archive = Path("/content") / archive_path.name

    if local_archive.exists():
        local_archive.unlink()

    with tarfile.open(local_archive, "w") as tar:
        tar.add(source_dir, arcname=source_dir.name)

    shutil.copy2(local_archive, archive_path)


def restore_tar_archive(archive_path, destination_parent=Path("/content")):
    archive_path = Path(archive_path)

    if not archive_path.exists():
        raise FileNotFoundError(f"Missing archive: {archive_path}")

    with tarfile.open(archive_path, "r") as tar:
        tar.extractall(destination_parent)


def read_papila_contour(path):
    points = []

    for line in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines():
        values = np.fromstring(line, sep=" ")
        if len(values) >= 2:
            points.append([values[0], values[1]])

    if len(points) < 3:
        raise ValueError(f"Invalid PAPILA contour file: {path}")

    return np.array(points, dtype=np.float32)


def rasterize_polygon(shape_hw, points):
    mask = np.zeros(shape_hw, dtype=np.uint8)
    polygon = np.round(points).astype(np.int32)
    cv2.fillPoly(mask, [polygon], 1)
    return mask


def resize_to_512(image):
    return cv2.resize(image, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_AREA)


def fill_hull_from_binary(mask):
    ys, xs = np.where(mask > 0)

    if len(xs) < 3:
        return np.zeros_like(mask, dtype=np.uint8)

    points = np.column_stack([xs, ys]).astype(np.int32)
    hull = cv2.convexHull(points)

    filled = np.zeros_like(mask, dtype=np.uint8)
    cv2.fillConvexPoly(filled, hull, 1)

    return filled


def reconstruct_riga_expert_masks(original_bgr, marked_bgr, diff_threshold=5):
    original_512 = resize_to_512(original_bgr)
    marked_512 = resize_to_512(marked_bgr)

    diff = cv2.absdiff(original_512, marked_512)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)

    line_mask = (diff_gray > diff_threshold).astype(np.uint8)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(line_mask, 8)

    components = []

    for label_id in range(1, num_labels):
        line_area = int(stats[label_id, cv2.CC_STAT_AREA])

        if line_area < 5:
            continue

        component = (labels == label_id).astype(np.uint8)
        filled = fill_hull_from_binary(component)
        filled_area = int(filled.sum())

        if filled_area < 20:
            continue

        components.append(
            {
                "line_area": line_area,
                "filled_area": filled_area,
                "mask": filled,
            }
        )

    components = sorted(components, key=lambda x: x["line_area"], reverse=True)

    if len(components) < 2:
        return None, None

    od_mask = components[0]["mask"]
    oc_mask = components[1]["mask"]
    oc_mask = (oc_mask & od_mask).astype(np.uint8)

    return od_mask, oc_mask


def consensus_mask(mask_list, min_votes=3):
    vote = np.stack(mask_list, axis=0).sum(axis=0)

    for threshold in [min_votes, 2, 1]:
        consensus = (vote >= threshold).astype(np.uint8)
        if consensus.sum() > 0:
            return consensus, threshold

    return np.zeros_like(mask_list[0], dtype=np.uint8), 0


def rebuild_refuge_standardized(raw_manifest_path, local_root):
    raw_manifest = pd.read_csv(raw_manifest_path)

    image_dir = local_root / "images_512"
    mask_dir = local_root / "masks_512"

    reset_directory(local_root)
    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for idx, row in raw_manifest.reset_index(drop=True).iterrows():
        image = cv2.imread(row["image_path"], cv2.IMREAD_COLOR)
        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise RuntimeError(f"Unreadable REFUGE image: {row['image_path']}")

        if mask is None:
            raise RuntimeError(f"Unreadable REFUGE mask: {row['mask_path']}")

        target_mask = np.zeros(mask.shape, dtype=np.uint8)
        target_mask[mask == 128] = 1
        target_mask[mask == 0] = 2

        image_512 = cv2.resize(image, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_AREA)
        mask_512 = cv2.resize(target_mask, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_NEAREST)

        image_512_path = image_dir / f"{row['sample_id']}.jpg"
        mask_512_path = mask_dir / f"{row['sample_id']}.png"

        cv2.imwrite(str(image_512_path), image_512)
        cv2.imwrite(str(mask_512_path), mask_512)

        rows.append(
            {
                "dataset": "REFUGE",
                "sample_id": row["sample_id"],
                "split_original": row["split_original"],
                "image_512_path": str(image_512_path),
                "mask_512_path": str(mask_512_path),
                "mask_values": "|".join(map(str, sorted(np.unique(mask_512).astype(int).tolist()))),
                "od_pixels_512": int((mask_512 == 1).sum()),
                "oc_pixels_512": int((mask_512 == 2).sum()),
            }
        )

        if (idx + 1) % 100 == 0:
            print(f"Processed REFUGE: {idx + 1}/{len(raw_manifest)}")

    return pd.DataFrame(rows)


def rebuild_papila_standardized(raw_manifest_path, local_root):
    raw_manifest = pd.read_csv(raw_manifest_path)

    image_dir = local_root / "images_512"
    mask_dir = local_root / "masks_512"

    reset_directory(local_root)
    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for idx, row in raw_manifest.reset_index(drop=True).iterrows():
        image = cv2.imread(row["image_path"], cv2.IMREAD_COLOR)

        if image is None:
            raise RuntimeError(f"Unreadable PAPILA image: {row['image_path']}")

        height, width = image.shape[:2]

        disc_exp1 = rasterize_polygon((height, width), read_papila_contour(row["disc_exp1_path"]))
        disc_exp2 = rasterize_polygon((height, width), read_papila_contour(row["disc_exp2_path"]))
        cup_exp1 = rasterize_polygon((height, width), read_papila_contour(row["cup_exp1_path"]))
        cup_exp2 = rasterize_polygon((height, width), read_papila_contour(row["cup_exp2_path"]))

        disc_mask = ((disc_exp1 + disc_exp2) >= 1).astype(np.uint8)
        cup_mask = ((cup_exp1 + cup_exp2) >= 1).astype(np.uint8)

        target_mask = np.zeros((height, width), dtype=np.uint8)
        target_mask[disc_mask == 1] = 1
        target_mask[cup_mask == 1] = 2

        image_512 = cv2.resize(image, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_AREA)
        mask_512 = cv2.resize(target_mask, (STANDARD_SIZE, STANDARD_SIZE), interpolation=cv2.INTER_NEAREST)

        image_512_path = image_dir / f"{row['sample_id']}.jpg"
        mask_512_path = mask_dir / f"{row['sample_id']}.png"

        cv2.imwrite(str(image_512_path), image_512)
        cv2.imwrite(str(mask_512_path), mask_512)

        rows.append(
            {
                "dataset": "PAPILA",
                "sample_id": row["sample_id"],
                "image_512_path": str(image_512_path),
                "mask_512_path": str(mask_512_path),
                "mask_values": "|".join(map(str, sorted(np.unique(mask_512).astype(int).tolist()))),
                "od_pixels_512": int((mask_512 == 1).sum()),
                "oc_pixels_512": int((mask_512 == 2).sum()),
            }
        )

        if (idx + 1) % 100 == 0:
            print(f"Processed PAPILA: {idx + 1}/{len(raw_manifest)}")

    return pd.DataFrame(rows)


def rebuild_riga_standardized(raw_manifest_path, local_root):
    raw_manifest = pd.read_csv(raw_manifest_path)

    image_dir = local_root / "images_512"
    mask_dir = local_root / "masks_512"

    reset_directory(local_root)
    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for idx, row in raw_manifest.reset_index(drop=True).iterrows():
        original = cv2.imread(row["original_path"], cv2.IMREAD_COLOR)

        if original is None:
            raise RuntimeError(f"Unreadable RIGA original: {row['original_path']}")

        image_512 = resize_to_512(original)

        od_masks = []
        oc_masks = []
        valid_experts = 0

        for expert_id in range(1, 7):
            marked_path = row[f"expert_marking_{expert_id}_path"]
            marked = cv2.imread(marked_path, cv2.IMREAD_COLOR)

            if marked is None:
                raise RuntimeError(f"Unreadable RIGA marking: {marked_path}")

            od_mask, oc_mask = reconstruct_riga_expert_masks(original, marked)

            if od_mask is not None and oc_mask is not None and od_mask.sum() > 0 and oc_mask.sum() > 0:
                od_masks.append(od_mask)
                oc_masks.append(oc_mask)
                valid_experts += 1

        if valid_experts == 0:
            od_consensus = np.zeros((STANDARD_SIZE, STANDARD_SIZE), dtype=np.uint8)
            oc_consensus = np.zeros((STANDARD_SIZE, STANDARD_SIZE), dtype=np.uint8)
            od_threshold = 0
            oc_threshold = 0
        else:
            od_consensus, od_threshold = consensus_mask(od_masks, min_votes=3)
            oc_consensus, oc_threshold = consensus_mask(oc_masks, min_votes=3)
            oc_consensus = (oc_consensus & od_consensus).astype(np.uint8)

        mask_512 = np.zeros((STANDARD_SIZE, STANDARD_SIZE), dtype=np.uint8)
        mask_512[od_consensus == 1] = 1
        mask_512[oc_consensus == 1] = 2

        image_512_path = image_dir / f"{row['sample_id']}.jpg"
        mask_512_path = mask_dir / f"{row['sample_id']}.png"

        cv2.imwrite(str(image_512_path), image_512)
        cv2.imwrite(str(mask_512_path), mask_512)

        rows.append(
            {
                "dataset": "RIGA",
                "sample_id": row["sample_id"],
                "source": row["source"],
                "subgroup": row["subgroup"],
                "image_512_path": str(image_512_path),
                "mask_512_path": str(mask_512_path),
                "mask_values": "|".join(map(str, sorted(np.unique(mask_512).astype(int).tolist()))),
                "od_pixels_512": int((mask_512 == 1).sum()),
                "oc_pixels_512": int((mask_512 == 2).sum()),
                "valid_experts": int(valid_experts),
                "od_threshold": int(od_threshold),
                "oc_threshold": int(oc_threshold),
            }
        )

        if (idx + 1) % 50 == 0:
            print(f"Processed RIGA: {idx + 1}/{len(raw_manifest)}")

    return pd.DataFrame(rows)


print("Defined restore, rebuild, and archive functions.")
print("Status: PASS")

In [ ]:
# Cell 3: Restore or rebuild standardized datasets and persist archives (RUN JUST ONE TIME)

rebuild_functions = {
    "REFUGE": rebuild_refuge_standardized,
    "PAPILA": rebuild_papila_standardized,
    "RIGA": rebuild_riga_standardized,
}

rebuild_rows = []

for dataset, spec in standardized_sources.items():
    local_root = spec["local_root"]
    image_dir = local_root / "images_512"
    mask_dir = local_root / "masks_512"
    archive_path = spec["archive_path"]
    raw_manifest_path = spec["raw_manifest"]
    standardized_manifest_path = spec["standardized_manifest"]
    expected_samples = spec["expected_samples"]

    local_images_before = len(list(image_dir.glob("*.jpg"))) if image_dir.exists() else 0
    local_masks_before = len(list(mask_dir.glob("*.png"))) if mask_dir.exists() else 0

    local_ready_before = (
        local_images_before == expected_samples and
        local_masks_before == expected_samples
    )

    if local_ready_before:
        action_taken = "used_existing_local"
        if standardized_manifest_path.exists():
            standardized_manifest = pd.read_csv(standardized_manifest_path)
        else:
            raise FileNotFoundError(f"Missing standardized manifest for {dataset}: {standardized_manifest_path}")

    elif archive_path.exists():
        if local_root.exists():
            shutil.rmtree(local_root)

        restore_tar_archive(archive_path, destination_parent=Path("/content"))
        action_taken = "restored_from_archive"

        if standardized_manifest_path.exists():
            standardized_manifest = pd.read_csv(standardized_manifest_path)
        else:
            raise FileNotFoundError(f"Missing standardized manifest after restore for {dataset}: {standardized_manifest_path}")

    else:
        if not raw_manifest_path.exists():
            raise FileNotFoundError(f"Missing raw manifest for {dataset}: {raw_manifest_path}")

        standardized_manifest = rebuild_functions[dataset](
            raw_manifest_path=raw_manifest_path,
            local_root=local_root,
        )

        standardized_manifest_path.parent.mkdir(parents=True, exist_ok=True)
        standardized_manifest.to_csv(standardized_manifest_path, index=False)

        results_manifest_path = (
            PATHS["results_04_od_segmentor_external_validation"] /
            f"notebook05_{dataset.lower()}_standardized_segmentation_manifest.csv"
        )
        standardized_manifest.to_csv(results_manifest_path, index=False)

        action_taken = "rebuilt_from_raw_manifest"

    local_images_after = len(list(image_dir.glob("*.jpg"))) if image_dir.exists() else 0
    local_masks_after = len(list(mask_dir.glob("*.png"))) if mask_dir.exists() else 0

    if local_images_after != expected_samples or local_masks_after != expected_samples:
        raise RuntimeError(
            f"{dataset} local count mismatch after action: "
            f"images={local_images_after}, masks={local_masks_after}, expected={expected_samples}"
        )

    if len(standardized_manifest) != expected_samples:
        raise RuntimeError(
            f"{dataset} standardized manifest mismatch: "
            f"rows={len(standardized_manifest)}, expected={expected_samples}"
        )

    empty_od = int((standardized_manifest["od_pixels_512"] == 0).sum())
    empty_oc = int((standardized_manifest["oc_pixels_512"] == 0).sum())

    if empty_od > 0 or empty_oc > 0:
        raise RuntimeError(f"{dataset} has empty OD/OC masks: empty_od={empty_od}, empty_oc={empty_oc}")

    if not archive_path.exists():
        create_tar_archive(local_root, archive_path)
        archive_action = "archive_created"
    else:
        archive_action = "archive_available"

    rebuild_rows.append(
        {
            "dataset": dataset,
            "action_taken": action_taken,
            "archive_action": archive_action,
            "local_images_512": local_images_after,
            "local_masks_512": local_masks_after,
            "manifest_rows": len(standardized_manifest),
            "empty_od_masks": empty_od,
            "empty_oc_masks": empty_oc,
            "archive_path": str(archive_path),
            "archive_size_mb": round(archive_path.stat().st_size / (1024 ** 2), 2),
        }
    )

standardized_restore_rebuild_log = pd.DataFrame(rebuild_rows)

restore_rebuild_log_csv = (
    ARCHIVE_DIR / "standardized_restore_rebuild_log.csv"
)

restore_rebuild_results_csv = (
    PATHS["results_04_od_segmentor_external_validation"] /
    "notebook05_standardized_restore_rebuild_log.csv"
)

standardized_restore_rebuild_log.to_csv(restore_rebuild_log_csv, index=False)
standardized_restore_rebuild_log.to_csv(restore_rebuild_results_csv, index=False)

print(
    standardized_restore_rebuild_log[
        [
            "dataset",
            "action_taken",
            "archive_action",
            "local_images_512",
            "local_masks_512",
            "manifest_rows",
            "empty_od_masks",
            "empty_oc_masks",
            "archive_size_mb",
        ]
    ].to_string(index=False)
)
print(f"Restore/rebuild log: {restore_rebuild_log_csv}")

if not standardized_restore_rebuild_log["archive_path"].apply(lambda x: Path(x).exists()).all():
    raise RuntimeError("One or more archives were not created.")

print("Status: PASS")

In [ ]:
# Cell 3: Restore standardized datasets from Drive archives (RUN FOR TRAINING)

restore_rows = []

for dataset, spec in standardized_sources.items():
    local_root = spec["local_root"]
    archive_path = spec["archive_path"]
    expected_samples = spec["expected_samples"]

    if not archive_path.exists():
        raise FileNotFoundError(
            f"Missing archive for {dataset}: {archive_path}. "
            "Do not rebuild in GPU runtime unless explicitly needed."
        )

    if local_root.exists():
        shutil.rmtree(local_root)

    restore_tar_archive(archive_path, destination_parent=Path("/content"))

    image_dir = local_root / "images_512"
    mask_dir = local_root / "masks_512"

    n_images = len(list(image_dir.glob("*.jpg"))) if image_dir.exists() else 0
    n_masks = len(list(mask_dir.glob("*.png"))) if mask_dir.exists() else 0

    if n_images != expected_samples or n_masks != expected_samples:
        raise RuntimeError(
            f"{dataset} restore failed: images={n_images}, masks={n_masks}, "
            f"expected={expected_samples}"
        )

    restore_rows.append(
        {
            "dataset": dataset,
            "archive_path": str(archive_path),
            "local_root": str(local_root),
            "n_images_512": n_images,
            "n_masks_512": n_masks,
            "archive_size_mb": round(archive_path.stat().st_size / (1024 ** 2), 2),
            "status": "restored",
        }
    )

standardized_restore_log = pd.DataFrame(restore_rows)

restore_log_csv = (
    ARCHIVE_DIR / "standardized_restore_only_log.csv"
)

restore_results_csv = (
    PATHS["results_04_od_segmentor_external_validation"] /
    "notebook05_standardized_restore_only_log.csv"
)

standardized_restore_log.to_csv(restore_log_csv, index=False)
standardized_restore_log.to_csv(restore_results_csv, index=False)

print(
    standardized_restore_log[
        ["dataset", "n_images_512", "n_masks_512", "archive_size_mb", "status"]
    ].to_string(index=False)
)
print(f"Restore log: {restore_log_csv}")
print("Status: PASS")

In [ ]:
# Cell 4: Load combined manifest and audit restored local files

RESULTS_05_DIR = JOURNAL_ROOT / "results" / "05_od_oc_segmentor_training"
RESULTS_05_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_SPLIT_MANIFEST_CSV = (
    PATHS["external_datasets"] /
    "standardized_segmentation" /
    "combined_standardized_segmentation_manifest_with_splits.csv"
)

if not COMBINED_SPLIT_MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Missing combined split manifest: {COMBINED_SPLIT_MANIFEST_CSV}")

segmentation_manifest = pd.read_csv(COMBINED_SPLIT_MANIFEST_CSV)

required_columns = [
    "dataset",
    "sample_id",
    "image_512_path",
    "mask_512_path",
    "od_pixels_512",
    "oc_pixels_512",
    "segmentation_split",
]

missing_columns = [col for col in required_columns if col not in segmentation_manifest.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

segmentation_manifest["image_exists"] = segmentation_manifest["image_512_path"].apply(lambda x: Path(x).exists())
segmentation_manifest["mask_exists"] = segmentation_manifest["mask_512_path"].apply(lambda x: Path(x).exists())

missing_images = int((~segmentation_manifest["image_exists"]).sum())
missing_masks = int((~segmentation_manifest["mask_exists"]).sum())
empty_od = int((segmentation_manifest["od_pixels_512"] == 0).sum())
empty_oc = int((segmentation_manifest["oc_pixels_512"] == 0).sum())
duplicate_sample_ids = int(segmentation_manifest["sample_id"].duplicated().sum())

runtime_summary = (
    segmentation_manifest.groupby(["dataset", "segmentation_split"])
    .agg(
        n_samples=("sample_id", "count"),
        images_exist=("image_exists", "sum"),
        masks_exist=("mask_exists", "sum"),
        empty_od=("od_pixels_512", lambda x: int((x == 0).sum())),
        empty_oc=("oc_pixels_512", lambda x: int((x == 0).sum())),
    )
    .reset_index()
)

# Lightweight mask-value audit
sample_audit_rows = []

for dataset in ["REFUGE", "PAPILA", "RIGA"]:
    dataset_df = segmentation_manifest[segmentation_manifest["dataset"] == dataset].copy()
    sample_df = dataset_df.groupby("segmentation_split", group_keys=False).head(3)

    for _, row in sample_df.iterrows():
        mask = cv2.imread(row["mask_512_path"], cv2.IMREAD_GRAYSCALE)

        if mask is None:
            raise RuntimeError(f"Unreadable mask: {row['mask_512_path']}")

        sample_audit_rows.append(
            {
                "dataset": row["dataset"],
                "segmentation_split": row["segmentation_split"],
                "sample_id": row["sample_id"],
                "mask_shape": str(mask.shape),
                "mask_values": "|".join(map(str, sorted(np.unique(mask).astype(int).tolist()))),
            }
        )

mask_sample_audit = pd.DataFrame(sample_audit_rows)

runtime_summary_csv = RESULTS_05_DIR / "notebook05_runtime_manifest_summary.csv"
mask_sample_audit_csv = RESULTS_05_DIR / "notebook05_mask_sample_audit.csv"

runtime_summary.to_csv(runtime_summary_csv, index=False)
mask_sample_audit.to_csv(mask_sample_audit_csv, index=False)

print(f"Manifest rows: {len(segmentation_manifest)}")
print(f"Missing images: {missing_images}")
print(f"Missing masks: {missing_masks}")
print(f"Empty OD masks: {empty_od}")
print(f"Empty OC masks: {empty_oc}")
print(f"Duplicate sample IDs: {duplicate_sample_ids}")
print(runtime_summary.to_string(index=False))
print(f"Runtime summary: {runtime_summary_csv}")

if len(segmentation_manifest) != 2438:
    raise RuntimeError(f"Expected 2438 samples, found {len(segmentation_manifest)}")

if missing_images > 0 or missing_masks > 0:
    raise RuntimeError("Missing restored local standardized files.")

if empty_od > 0 or empty_oc > 0:
    raise RuntimeError("Empty OD/OC masks found.")

if duplicate_sample_ids > 0:
    raise RuntimeError("Duplicate sample IDs found.")

print("Status: PASS")

In [ ]:
# Cell 5: Configure training runtime and experiment paths

import random
import time

import torch

from sklearn.metrics import confusion_matrix

SEED = int(PROJECT_CONFIG["seed"])
SEGMENTATION_DATASETS = ["REFUGE", "PAPILA", "RIGA"]

NUM_CLASSES = 3
CLASS_NAMES = {
    0: "background",
    1: "optic_disc",
    2: "optic_cup",
}

INPUT_SIZE = 512
BATCH_SIZE = 8
NUM_WORKERS = 2

MAX_EPOCHS = 120
EARLY_STOP_PATIENCE = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

PRIMARY_METRIC = "mean_dice_od_oc"
USE_AMP = torch.cuda.is_available()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_ROOT = JOURNAL_ROOT / "models" / "od_oc_segmentors"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

TRAINING_PLAN_CSV = (
    PATHS["external_datasets"] /
    "standardized_segmentation" /
    "segmentation_training_plan.csv"
)

EVALUATION_PLAN_CSV = (
    PATHS["external_datasets"] /
    "standardized_segmentation" /
    "cross_dataset_evaluation_plan.csv"
)

if not TRAINING_PLAN_CSV.exists():
    raise FileNotFoundError(f"Missing training plan: {TRAINING_PLAN_CSV}")

if not EVALUATION_PLAN_CSV.exists():
    raise FileNotFoundError(f"Missing evaluation plan: {EVALUATION_PLAN_CSV}")

segmentation_training_plan = pd.read_csv(TRAINING_PLAN_CSV)
cross_dataset_evaluation_plan = pd.read_csv(EVALUATION_PLAN_CSV)

for dataset in SEGMENTATION_DATASETS:
    segmentor_dir = MODEL_ROOT / f"S_{dataset}"
    (segmentor_dir / "checkpoints").mkdir(parents=True, exist_ok=True)
    (segmentor_dir / "logs").mkdir(parents=True, exist_ok=True)
    (segmentor_dir / "predictions").mkdir(parents=True, exist_ok=True)

runtime_config = {
    "seed": SEED,
    "device": str(DEVICE),
    "cuda_available": bool(torch.cuda.is_available()),
    "torch_version": torch.__version__,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "input_size": INPUT_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "max_epochs": MAX_EPOCHS,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "primary_metric": PRIMARY_METRIC,
    "use_amp": bool(USE_AMP),
    "model_root": str(MODEL_ROOT),
}

runtime_config_json = RESULTS_05_DIR / "notebook05_runtime_config.json"

with open(runtime_config_json, "w", encoding="utf-8") as f:
    json.dump(runtime_config, f, indent=2)

print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"AMP enabled: {USE_AMP}")
print(f"Training datasets: {SEGMENTATION_DATASETS}")
print(f"Segmentors planned: {len(segmentation_training_plan)}")
print(f"Evaluation pairs planned: {len(cross_dataset_evaluation_plan)}")
print(f"Model root: {MODEL_ROOT}")
print(f"Runtime config: {runtime_config_json}")

if len(segmentation_training_plan) != 3:
    raise RuntimeError("Expected 3 dataset-specific segmentors.")

if len(cross_dataset_evaluation_plan) != 9:
    raise RuntimeError("Expected 9 cross-dataset evaluation pairs.")

print("Status: PASS")

In [ ]:
# Cell 6: Define segmentation dataset and dataloader smoke test

from torch.utils.data import Dataset, DataLoader

class ODOCSegmentationDataset(Dataset):
    def __init__(self, dataframe, augment=False):
        self.df = dataframe.reset_index(drop=True).copy()
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = cv2.imread(row["image_512_path"], cv2.IMREAD_COLOR)
        mask = cv2.imread(row["mask_512_path"], cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise RuntimeError(f"Unreadable image: {row['image_512_path']}")

        if mask is None:
            raise RuntimeError(f"Unreadable mask: {row['mask_512_path']}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if image.shape[:2] != (INPUT_SIZE, INPUT_SIZE):
            raise RuntimeError(f"Unexpected image shape: {image.shape} for {row['sample_id']}")

        if mask.shape != (INPUT_SIZE, INPUT_SIZE):
            raise RuntimeError(f"Unexpected mask shape: {mask.shape} for {row['sample_id']}")

        if self.augment:
            if np.random.rand() < 0.5:
                image = np.fliplr(image).copy()
                mask = np.fliplr(mask).copy()

        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))

        image_tensor = torch.from_numpy(np.ascontiguousarray(image)).float()
        mask_tensor = torch.from_numpy(np.ascontiguousarray(mask)).long()

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "sample_id": row["sample_id"],
            "dataset": row["dataset"],
        }


def make_loader(dataframe, batch_size=BATCH_SIZE, shuffle=False, augment=False):
    dataset = ODOCSegmentationDataset(
        dataframe=dataframe,
        augment=augment,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
        drop_last=False,
    )


def get_dataset_split(dataset_name, split_name):
    subset = segmentation_manifest[
        (segmentation_manifest["dataset"] == dataset_name) &
        (segmentation_manifest["segmentation_split"] == split_name)
    ].copy()

    if len(subset) == 0:
        raise RuntimeError(f"Empty subset: dataset={dataset_name}, split={split_name}")

    return subset.reset_index(drop=True)


loader_smoke_rows = []

for dataset_name in SEGMENTATION_DATASETS:
    train_df = get_dataset_split(dataset_name, "train")
    val_df = get_dataset_split(dataset_name, "val")
    test_df = get_dataset_split(dataset_name, "test")

    train_loader = make_loader(train_df, shuffle=True, augment=True)
    val_loader = make_loader(val_df, shuffle=False, augment=False)
    test_loader = make_loader(test_df, shuffle=False, augment=False)

    batch = next(iter(train_loader))

    images = batch["image"]
    masks = batch["mask"]
    mask_values = sorted(torch.unique(masks).cpu().numpy().astype(int).tolist())

    loader_smoke_rows.append(
        {
            "dataset": dataset_name,
            "train_samples": len(train_df),
            "val_samples": len(val_df),
            "test_samples": len(test_df),
            "batch_image_shape": str(tuple(images.shape)),
            "batch_mask_shape": str(tuple(masks.shape)),
            "batch_mask_values": "|".join(map(str, mask_values)),
        }
    )

loader_smoke_test = pd.DataFrame(loader_smoke_rows)

loader_smoke_csv = RESULTS_05_DIR / "notebook05_dataloader_smoke_test.csv"
loader_smoke_test.to_csv(loader_smoke_csv, index=False)

print(loader_smoke_test.to_string(index=False))
print(f"Dataloader smoke test: {loader_smoke_csv}")

expected_values = {0, 1, 2}

for values in loader_smoke_test["batch_mask_values"]:
    parsed_values = set(int(v) for v in values.split("|"))
    if not parsed_values.issubset(expected_values):
        raise RuntimeError(f"Unexpected mask values: {parsed_values}")

print("Status: PASS")

In [ ]:
# Cell 7: Define candidate segmentation models and run smoke tests

import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        )

    def forward(self, x):
        return F.relu(self.conv(x) + self.skip(x), inplace=True)


class AttentionGate(nn.Module):
    def __init__(self, gate_channels, skip_channels, inter_channels):
        super().__init__()

        self.gate_proj = nn.Sequential(
            nn.Conv2d(gate_channels, inter_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(inter_channels),
        )

        self.skip_proj = nn.Sequential(
            nn.Conv2d(skip_channels, inter_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(inter_channels),
        )

        self.psi = nn.Sequential(
            nn.Conv2d(inter_channels, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, gate, skip):
        attention = F.relu(self.gate_proj(gate) + self.skip_proj(skip), inplace=True)
        attention = self.psi(attention)
        return skip * attention


class CompactUNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, base_channels=24):
        super().__init__()

        self.enc1 = ConvBlock(in_channels, base_channels)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.enc3 = ConvBlock(base_channels * 2, base_channels * 4)
        self.enc4 = ConvBlock(base_channels * 4, base_channels * 8)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base_channels * 8, base_channels * 16)

        self.up4 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, 2, 2)
        self.dec4 = ConvBlock(base_channels * 16, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, 2)
        self.dec3 = ConvBlock(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, 2)
        self.dec2 = ConvBlock(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, 2)
        self.dec1 = ConvBlock(base_channels * 2, base_channels)

        self.head = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.head(d1)


class ResidualUNetLite(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, base_channels=24):
        super().__init__()

        self.enc1 = ResidualBlock(in_channels, base_channels)
        self.enc2 = ResidualBlock(base_channels, base_channels * 2)
        self.enc3 = ResidualBlock(base_channels * 2, base_channels * 4)
        self.enc4 = ResidualBlock(base_channels * 4, base_channels * 8)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ResidualBlock(base_channels * 8, base_channels * 16)

        self.up4 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, 2, 2)
        self.dec4 = ResidualBlock(base_channels * 16, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, 2)
        self.dec3 = ResidualBlock(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, 2)
        self.dec2 = ResidualBlock(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, 2)
        self.dec1 = ResidualBlock(base_channels * 2, base_channels)

        self.head = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.head(d1)


class AttentionUNetLite(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, base_channels=24):
        super().__init__()

        self.enc1 = ConvBlock(in_channels, base_channels)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.enc3 = ConvBlock(base_channels * 2, base_channels * 4)
        self.enc4 = ConvBlock(base_channels * 4, base_channels * 8)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base_channels * 8, base_channels * 16)

        self.up4 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, 2, 2)
        self.att4 = AttentionGate(base_channels * 8, base_channels * 8, base_channels * 4)
        self.dec4 = ConvBlock(base_channels * 16, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, 2)
        self.att3 = AttentionGate(base_channels * 4, base_channels * 4, base_channels * 2)
        self.dec3 = ConvBlock(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, 2)
        self.att2 = AttentionGate(base_channels * 2, base_channels * 2, base_channels)
        self.dec2 = ConvBlock(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, 2)
        self.att1 = AttentionGate(base_channels, base_channels, max(base_channels // 2, 1))
        self.dec1 = ConvBlock(base_channels * 2, base_channels)

        self.head = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        u4 = self.up4(b)
        d4 = self.dec4(torch.cat([u4, self.att4(u4, e4)], dim=1))

        u3 = self.up3(d4)
        d3 = self.dec3(torch.cat([u3, self.att3(u3, e3)], dim=1))

        u2 = self.up2(d3)
        d2 = self.dec2(torch.cat([u2, self.att2(u2, e2)], dim=1))

        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, self.att1(u1, e1)], dim=1))

        return self.head(d1)


class ASPPBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.branch1 = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.branch2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=2, dilation=2)
        self.branch3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=4, dilation=4)
        self.branch4 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=8, dilation=8)

        self.project = nn.Sequential(
            nn.Conv2d(out_channels * 4, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = torch.cat(
            [
                self.branch1(x),
                self.branch2(x),
                self.branch3(x),
                self.branch4(x),
            ],
            dim=1,
        )

        return self.project(x)


class DeepLabLite(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, base_channels=32):
        super().__init__()

        self.stem = ConvBlock(in_channels, base_channels)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.enc3 = ConvBlock(base_channels * 2, base_channels * 4)
        self.enc4 = ConvBlock(base_channels * 4, base_channels * 8)

        self.pool = nn.MaxPool2d(2)
        self.aspp = ASPPBlock(base_channels * 8, base_channels * 4)

        self.decoder = nn.Sequential(
            ConvBlock(base_channels * 4 + base_channels, base_channels * 2),
            nn.Conv2d(base_channels * 2, num_classes, kernel_size=1),
        )

    def forward(self, x):
        input_size = x.shape[-2:]

        low = self.stem(x)
        x = self.enc2(self.pool(low))
        x = self.enc3(self.pool(x))
        x = self.enc4(self.pool(x))
        x = self.aspp(x)

        x = F.interpolate(x, size=low.shape[-2:], mode="bilinear", align_corners=False)
        x = self.decoder(torch.cat([x, low], dim=1))
        x = F.interpolate(x, size=input_size, mode="bilinear", align_corners=False)

        return x


class FPNLite(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, base_channels=32):
        super().__init__()

        self.enc1 = ConvBlock(in_channels, base_channels)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.enc3 = ConvBlock(base_channels * 2, base_channels * 4)
        self.enc4 = ConvBlock(base_channels * 4, base_channels * 8)

        self.pool = nn.MaxPool2d(2)

        self.lat1 = nn.Conv2d(base_channels, base_channels, kernel_size=1)
        self.lat2 = nn.Conv2d(base_channels * 2, base_channels, kernel_size=1)
        self.lat3 = nn.Conv2d(base_channels * 4, base_channels, kernel_size=1)
        self.lat4 = nn.Conv2d(base_channels * 8, base_channels, kernel_size=1)

        self.smooth = ConvBlock(base_channels, base_channels)
        self.head = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        p4 = self.lat4(e4)
        p3 = self.lat3(e3) + F.interpolate(p4, size=e3.shape[-2:], mode="bilinear", align_corners=False)
        p2 = self.lat2(e2) + F.interpolate(p3, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        p1 = self.lat1(e1) + F.interpolate(p2, size=e1.shape[-2:], mode="bilinear", align_corners=False)

        out = self.smooth(p1)
        return self.head(out)


def soft_dice_loss(logits, targets, classes=(1, 2), eps=1e-7):
    probs = torch.softmax(logits, dim=1)
    total_loss = 0.0

    for class_idx in classes:
        pred = probs[:, class_idx, :, :]
        true = (targets == class_idx).float()

        intersection = (pred * true).sum(dim=(1, 2))
        denominator = pred.sum(dim=(1, 2)) + true.sum(dim=(1, 2))

        dice = (2.0 * intersection + eps) / (denominator + eps)
        total_loss += 1.0 - dice.mean()

    return total_loss / len(classes)


class CEDiceLoss(nn.Module):
    def __init__(self, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        dice_loss = soft_dice_loss(logits, targets, classes=(1, 2))
        return self.ce_weight * ce_loss + self.dice_weight * dice_loss


def dice_score_per_class(logits, targets, num_classes=NUM_CLASSES, eps=1e-7):
    preds = torch.argmax(logits, dim=1)
    scores = {}

    for class_idx in range(num_classes):
        pred = (preds == class_idx).float()
        true = (targets == class_idx).float()

        intersection = (pred * true).sum()
        denominator = pred.sum() + true.sum()

        scores[class_idx] = ((2.0 * intersection + eps) / (denominator + eps)).detach()

    return scores


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


MODEL_REGISTRY = {
    "CompactUNet": lambda: CompactUNet(in_channels=3, num_classes=NUM_CLASSES, base_channels=24),
    "ResidualUNetLite": lambda: ResidualUNetLite(in_channels=3, num_classes=NUM_CLASSES, base_channels=24),
    "AttentionUNetLite": lambda: AttentionUNetLite(in_channels=3, num_classes=NUM_CLASSES, base_channels=24),
    "DeepLabLite": lambda: DeepLabLite(in_channels=3, num_classes=NUM_CLASSES, base_channels=32),
    "FPNLite": lambda: FPNLite(in_channels=3, num_classes=NUM_CLASSES, base_channels=32),
}

criterion = CEDiceLoss(ce_weight=0.5, dice_weight=0.5)

smoke_df = get_dataset_split("REFUGE", "train").head(2)
smoke_loader = make_loader(smoke_df, batch_size=2, shuffle=False, augment=False)
smoke_batch = next(iter(smoke_loader))

images = smoke_batch["image"].to(DEVICE, non_blocking=True)
masks = smoke_batch["mask"].to(DEVICE, non_blocking=True)

model_smoke_rows = []

for model_name, model_fn in MODEL_REGISTRY.items():
    torch.cuda.empty_cache()

    model = model_fn().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    model.train()
    optimizer.zero_grad(set_to_none=True)

    start_time = time.time()

    with torch.cuda.amp.autocast(enabled=USE_AMP):
        logits = model(images)
        loss = criterion(logits, masks)

    loss.backward()
    optimizer.step()

    elapsed_sec = time.time() - start_time
    dice_scores = dice_score_per_class(logits.detach(), masks, num_classes=NUM_CLASSES)

    model_smoke_rows.append(
        {
            "model_name": model_name,
            "trainable_parameters": int(count_trainable_parameters(model)),
            "input_shape": str(tuple(images.shape)),
            "logits_shape": str(tuple(logits.shape)),
            "loss": float(loss.detach().cpu()),
            "dice_od": float(dice_scores[1].cpu()),
            "dice_oc": float(dice_scores[2].cpu()),
            "elapsed_sec": round(elapsed_sec, 4),
            "status": "pass",
        }
    )

    del model, optimizer, logits, loss
    torch.cuda.empty_cache()

model_smoke_test = pd.DataFrame(model_smoke_rows)

model_registry_csv = RESULTS_05_DIR / "notebook05_model_registry_smoke_test.csv"
model_smoke_test.to_csv(model_registry_csv, index=False)

print(model_smoke_test[["model_name", "trainable_parameters", "logits_shape", "loss", "elapsed_sec", "status"]].to_string(index=False))
print(f"Model smoke test: {model_registry_csv}")

if len(model_smoke_test) != 5:
    raise RuntimeError("Expected 5 candidate models.")

if not np.isfinite(model_smoke_test["loss"]).all():
    raise RuntimeError("Non-finite loss found in model smoke test.")

if not model_smoke_test["logits_shape"].eq(str((2, NUM_CLASSES, INPUT_SIZE, INPUT_SIZE))).all():
    raise RuntimeError("Unexpected logits shape in one or more models.")

print("Status: PASS")

In [ ]:
# Cell 8: Define training, validation, metrics, and checkpoint utilities

def build_model(model_name):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model: {model_name}")
    return MODEL_REGISTRY[model_name]()


def compute_segmentation_metrics_from_logits(logits, masks, eps=1e-7):
    preds = torch.argmax(logits, dim=1)

    metrics = {}

    for class_idx, class_name in [(1, "od"), (2, "oc")]:
        pred_mask = (preds == class_idx)
        true_mask = (masks == class_idx)

        intersection = (pred_mask & true_mask).sum().float()
        union = (pred_mask | true_mask).sum().float()
        pred_sum = pred_mask.sum().float()
        true_sum = true_mask.sum().float()

        dice = (2.0 * intersection + eps) / (pred_sum + true_sum + eps)
        iou = (intersection + eps) / (union + eps)

        metrics[f"dice_{class_name}"] = float(dice.detach().cpu())
        metrics[f"iou_{class_name}"] = float(iou.detach().cpu())

    metrics["mean_dice_od_oc"] = float(
        0.5 * (metrics["dice_od"] + metrics["dice_oc"])
    )

    metrics["mean_iou_od_oc"] = float(
        0.5 * (metrics["iou_od"] + metrics["iou_oc"])
    )

    return metrics


def aggregate_metric_rows(metric_rows):
    if len(metric_rows) == 0:
        raise ValueError("No metric rows to aggregate.")

    df = pd.DataFrame(metric_rows)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    return {
        col: float(df[col].mean())
        for col in numeric_cols
    }


def amp_autocast_context():
    return torch.amp.autocast(
        device_type="cuda",
        enabled=USE_AMP,
    )


def make_grad_scaler():
    return torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()

    loss_rows = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with amp_autocast_context():
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_rows.append(
            {
                "loss": float(loss.detach().cpu()),
            }
        )

    return aggregate_metric_rows(loss_rows)


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    metric_rows = []

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)

        with amp_autocast_context():
            logits = model(images)
            loss = criterion(logits, masks)

        batch_metrics = compute_segmentation_metrics_from_logits(logits, masks)
        batch_metrics["loss"] = float(loss.detach().cpu())

        metric_rows.append(batch_metrics)

    return aggregate_metric_rows(metric_rows)


def get_segmentor_paths(segmentor_id, model_name=None):
    if model_name is None:
        run_dir = MODEL_ROOT / segmentor_id
    else:
        run_dir = MODEL_ROOT / "architecture_screening" / segmentor_id / model_name

    checkpoint_dir = run_dir / "checkpoints"
    log_dir = run_dir / "logs"

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    return {
        "run_dir": run_dir,
        "checkpoint_dir": checkpoint_dir,
        "log_dir": log_dir,
        "latest_checkpoint": checkpoint_dir / "latest_checkpoint.pt",
        "best_checkpoint": checkpoint_dir / "best_checkpoint.pt",
        "training_log_csv": log_dir / "training_log.csv",
        "config_json": run_dir / "run_config.json",
    }


def save_checkpoint(
    checkpoint_path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    best_metric,
    config,
    history,
):
    payload = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "best_metric": float(best_metric),
        "config": config,
        "history": history,
    }

    torch.save(payload, checkpoint_path)


def load_checkpoint_if_available(
    checkpoint_path,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    device=DEVICE,
):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        return 0, -np.inf, []

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and checkpoint.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = int(checkpoint["epoch"]) + 1
    best_metric = float(checkpoint.get("best_metric", -np.inf))
    history = checkpoint.get("history", [])

    return start_epoch, best_metric, history


def run_training_loop(
    model_name,
    train_dataset_name,
    train_df,
    val_df,
    max_epochs,
    patience,
    screening_mode,
):
    segmentor_id = f"S_{train_dataset_name}"
    paths = get_segmentor_paths(
        segmentor_id=segmentor_id,
        model_name=model_name if screening_mode else None,
    )

    config = {
        "model_name": model_name,
        "segmentor_id": segmentor_id,
        "train_dataset": train_dataset_name,
        "screening_mode": bool(screening_mode),
        "seed": SEED,
        "input_size": INPUT_SIZE,
        "batch_size": BATCH_SIZE,
        "max_epochs": int(max_epochs),
        "patience": int(patience),
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "primary_metric": PRIMARY_METRIC,
    }

    with open(paths["config_json"], "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    train_loader = make_loader(train_df, shuffle=True, augment=True)
    val_loader = make_loader(val_df, shuffle=False, augment=False)

    model = build_model(model_name).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=max(2, patience // 3),
    )

    scaler = make_grad_scaler()

    start_epoch, best_metric, history = load_checkpoint_if_available(
        paths["latest_checkpoint"],
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=DEVICE,
    )

    epochs_without_improvement = 0

    for epoch in range(start_epoch, max_epochs):
        epoch_start = time.time()

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
        )

        val_metrics = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=DEVICE,
        )

        current_metric = float(val_metrics[PRIMARY_METRIC])
        scheduler.step(current_metric)

        improved = current_metric > best_metric

        if improved:
            best_metric = current_metric
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        epoch_row = {
            "epoch": int(epoch),
            "model_name": model_name,
            "train_dataset": train_dataset_name,
            "train_loss": float(train_metrics["loss"]),
            "val_loss": float(val_metrics["loss"]),
            "val_dice_od": float(val_metrics["dice_od"]),
            "val_dice_oc": float(val_metrics["dice_oc"]),
            "val_mean_dice_od_oc": float(val_metrics["mean_dice_od_oc"]),
            "val_iou_od": float(val_metrics["iou_od"]),
            "val_iou_oc": float(val_metrics["iou_oc"]),
            "val_mean_iou_od_oc": float(val_metrics["mean_iou_od_oc"]),
            "best_metric": float(best_metric),
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
            "elapsed_sec": round(time.time() - epoch_start, 2),
            "improved": bool(improved),
        }

        history.append(epoch_row)

        pd.DataFrame(history).to_csv(paths["training_log_csv"], index=False)

        save_checkpoint(
            checkpoint_path=paths["latest_checkpoint"],
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch,
            best_metric=best_metric,
            config=config,
            history=history,
        )

        if improved:
            save_checkpoint(
                checkpoint_path=paths["best_checkpoint"],
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                best_metric=best_metric,
                config=config,
                history=history,
            )

        print(
            f"{model_name} | {train_dataset_name} | "
            f"epoch {epoch + 1}/{max_epochs} | "
            f"val_mean_dice={current_metric:.4f} | "
            f"best={best_metric:.4f} | "
            f"elapsed={epoch_row['elapsed_sec']}s"
        )

        if epochs_without_improvement >= patience:
            break

    best_row = max(history, key=lambda x: x["val_mean_dice_od_oc"])

    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

    return {
        "model_name": model_name,
        "train_dataset": train_dataset_name,
        "best_epoch": int(best_row["epoch"]),
        "best_val_mean_dice_od_oc": float(best_row["val_mean_dice_od_oc"]),
        "best_val_dice_od": float(best_row["val_dice_od"]),
        "best_val_dice_oc": float(best_row["val_dice_oc"]),
        "epochs_ran": len(history),
        "best_checkpoint": str(paths["best_checkpoint"]),
        "latest_checkpoint": str(paths["latest_checkpoint"]),
        "training_log": str(paths["training_log_csv"]),
    }


print("Training utilities ready.")
print("Status: PASS")

In [ ]:
# Cell 8B: Configure CUDA determinism warnings for segmentation training

import warnings

torch.use_deterministic_algorithms(True, warn_only=True)

warnings.filterwarnings(
    "ignore",
    message=".*nll_loss2d_forward_out_cuda_template does not have a deterministic implementation.*",
    category=UserWarning,
)

determinism_note = {
    "torch_deterministic_algorithms": True,
    "warn_only": True,
    "cuda_cross_entropy_determinism": "PyTorch warns that CUDA nll_loss2d is not strictly deterministic; training remains seed-controlled and checkpointed.",
    "decision": "Proceed with CE+Dice loss on GPU for practical reproducible training.",
}

determinism_note_json = RESULTS_05_DIR / "notebook05_determinism_note.json"

with open(determinism_note_json, "w", encoding="utf-8") as f:
    json.dump(determinism_note, f, indent=2)

print(f"Determinism note: {determinism_note_json}")
print("Status: PASS")

In [ ]:
# Cell 9: Short architecture screening across dataset-specific segmentors

SCREENING_MODELS = [
    "CompactUNet",
    "ResidualUNetLite",
    "AttentionUNetLite",
    "DeepLabLite",
    "FPNLite",
]

SCREENING_DATASETS = [
    "REFUGE",
    "PAPILA",
    "RIGA",
]

SCREENING_EPOCHS = 5
SCREENING_PATIENCE = 3

screening_rows = []

for dataset_name in SCREENING_DATASETS:
    train_df = get_dataset_split(dataset_name, "train")
    val_df = get_dataset_split(dataset_name, "val")

    for model_name in SCREENING_MODELS:
        result = run_training_loop(
            model_name=model_name,
            train_dataset_name=dataset_name,
            train_df=train_df,
            val_df=val_df,
            max_epochs=SCREENING_EPOCHS,
            patience=SCREENING_PATIENCE,
            screening_mode=True,
        )

        screening_rows.append(result)

        screening_results_live = pd.DataFrame(screening_rows)
        screening_results_live.to_csv(
            RESULTS_05_DIR / "notebook05_architecture_screening_results_live.csv",
            index=False,
        )

architecture_screening_results = pd.DataFrame(screening_rows)

architecture_screening_results_csv = (
    RESULTS_05_DIR / "notebook05_architecture_screening_results.csv"
)

architecture_screening_results.to_csv(architecture_screening_results_csv, index=False)

architecture_screening_summary = (
    architecture_screening_results.groupby("model_name")
    .agg(
        mean_val_dice=("best_val_mean_dice_od_oc", "mean"),
        mean_val_dice_od=("best_val_dice_od", "mean"),
        mean_val_dice_oc=("best_val_dice_oc", "mean"),
        min_val_dice=("best_val_mean_dice_od_oc", "min"),
        max_val_dice=("best_val_mean_dice_od_oc", "max"),
        total_epochs=("epochs_ran", "sum"),
    )
    .reset_index()
    .sort_values(["mean_val_dice", "min_val_dice"], ascending=False)
)

architecture_screening_summary_csv = (
    RESULTS_05_DIR / "notebook05_architecture_screening_summary.csv"
)

architecture_screening_summary.to_csv(architecture_screening_summary_csv, index=False)

print("Architecture screening summary:")
print(architecture_screening_summary.to_string(index=False))
print(f"Screening results: {architecture_screening_results_csv}")
print(f"Screening summary: {architecture_screening_summary_csv}")

if len(architecture_screening_results) != len(SCREENING_MODELS) * len(SCREENING_DATASETS):
    raise RuntimeError("Unexpected number of architecture screening runs.")

print("Status: PASS")

In [ ]:
# Cell 10: Select architectures for full training

SCREENING_RESULTS_CSV = RESULTS_05_DIR / "notebook05_architecture_screening_results.csv"
SCREENING_SUMMARY_CSV = RESULTS_05_DIR / "notebook05_architecture_screening_summary.csv"

if not SCREENING_RESULTS_CSV.exists():
    raise FileNotFoundError(f"Missing screening results: {SCREENING_RESULTS_CSV}")

if not SCREENING_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing screening summary: {SCREENING_SUMMARY_CSV}")

architecture_screening_results = pd.read_csv(SCREENING_RESULTS_CSV)
architecture_screening_summary = pd.read_csv(SCREENING_SUMMARY_CSV)

required_summary_columns = [
    "model_name",
    "mean_val_dice",
    "mean_val_dice_od",
    "mean_val_dice_oc",
    "min_val_dice",
    "max_val_dice",
    "total_epochs",
]

missing_columns = [
    col for col in required_summary_columns
    if col not in architecture_screening_summary.columns
]

if missing_columns:
    raise ValueError(f"Missing screening summary columns: {missing_columns}")

architecture_screening_summary = architecture_screening_summary.sort_values(
    ["mean_val_dice", "min_val_dice"],
    ascending=False,
).reset_index(drop=True)

selected_models = architecture_screening_summary.head(2)["model_name"].tolist()
backup_model = architecture_screening_summary.iloc[2]["model_name"]

selection_rows = []

for rank, row in architecture_screening_summary.iterrows():
    model_name = row["model_name"]

    if model_name in selected_models:
        decision = "selected_for_full_training"
    elif model_name == backup_model:
        decision = "backup_candidate"
    else:
        decision = "not_selected_after_screening"

    selection_rows.append(
        {
            "rank": int(rank + 1),
            "model_name": model_name,
            "decision": decision,
            "mean_val_dice": float(row["mean_val_dice"]),
            "mean_val_dice_od": float(row["mean_val_dice_od"]),
            "mean_val_dice_oc": float(row["mean_val_dice_oc"]),
            "min_val_dice": float(row["min_val_dice"]),
            "max_val_dice": float(row["max_val_dice"]),
            "total_epochs": int(row["total_epochs"]),
        }
    )

architecture_selection = pd.DataFrame(selection_rows)

selection_config = {
    "selection_stage": "short_architecture_screening",
    "screening_epochs": int(SCREENING_EPOCHS),
    "screening_datasets": SCREENING_DATASETS,
    "screening_models": SCREENING_MODELS,
    "ranking_metric": "mean_val_dice_od_oc",
    "tie_breaker": "min_val_dice",
    "selected_models": selected_models,
    "backup_model": backup_model,
    "decision_rationale": {
        selected_models[0]: "Best mean validation Dice across REFUGE, PAPILA and RIGA in short screening.",
        selected_models[1]: "Second-best mean validation Dice with strong OD performance and lower computational cost than DeepLabLite.",
        backup_model: "Third-best candidate kept as fallback if selected models fail during full training or cross-dataset evaluation.",
    },
}

architecture_selection_csv = RESULTS_05_DIR / "notebook05_selected_architectures.csv"
architecture_selection_json = RESULTS_05_DIR / "notebook05_selected_architectures_config.json"

architecture_selection.to_csv(architecture_selection_csv, index=False)

with open(architecture_selection_json, "w", encoding="utf-8") as f:
    json.dump(selection_config, f, indent=2)

print("Selected models for full training:")
print(architecture_selection.to_string(index=False))
print(f"Selection CSV: {architecture_selection_csv}")
print(f"Selection config: {architecture_selection_json}")

if selected_models != ["DeepLabLite", "ResidualUNetLite"]:
    raise RuntimeError(f"Unexpected selected models: {selected_models}")

print("Status: PASS")

In [ ]:
# Cell 11: Configure full training plan and full-training runner

SELECTED_ARCHITECTURES_CSV = RESULTS_05_DIR / "notebook05_selected_architectures.csv"

if not SELECTED_ARCHITECTURES_CSV.exists():
    raise FileNotFoundError(f"Missing selected architectures CSV: {SELECTED_ARCHITECTURES_CSV}")

architecture_selection = pd.read_csv(SELECTED_ARCHITECTURES_CSV)

FULL_TRAINING_MODELS = architecture_selection.loc[
    architecture_selection["decision"] == "selected_for_full_training",
    "model_name",
].tolist()

FULL_TRAINING_DATASETS = ["REFUGE", "PAPILA", "RIGA"]

FULL_MAX_EPOCHS = 80
FULL_EARLY_STOP_PATIENCE = 15

FULL_TRAINING_ROOT = MODEL_ROOT / "full_training"
FULL_TRAINING_ROOT.mkdir(parents=True, exist_ok=True)

full_plan_rows = []

for model_name in FULL_TRAINING_MODELS:
    for dataset_name in FULL_TRAINING_DATASETS:
        train_df = get_dataset_split(dataset_name, "train")
        val_df = get_dataset_split(dataset_name, "val")
        test_df = get_dataset_split(dataset_name, "test")

        run_id = f"{model_name}__S_{dataset_name}"
        run_dir = FULL_TRAINING_ROOT / model_name / f"S_{dataset_name}"

        full_plan_rows.append(
            {
                "run_id": run_id,
                "model_name": model_name,
                "segmentor_id": f"S_{dataset_name}",
                "train_dataset": dataset_name,
                "train_samples": len(train_df),
                "val_samples": len(val_df),
                "internal_test_samples": len(test_df),
                "max_epochs": FULL_MAX_EPOCHS,
                "early_stop_patience": FULL_EARLY_STOP_PATIENCE,
                "primary_metric": PRIMARY_METRIC,
                "run_dir": str(run_dir),
                "latest_checkpoint": str(run_dir / "checkpoints" / "latest_checkpoint.pt"),
                "best_checkpoint": str(run_dir / "checkpoints" / "best_checkpoint.pt"),
                "training_log": str(run_dir / "logs" / "training_log.csv"),
            }
        )

full_training_plan = pd.DataFrame(full_plan_rows)

full_training_plan_csv = RESULTS_05_DIR / "notebook05_full_training_plan.csv"
full_training_plan.to_csv(full_training_plan_csv, index=False)

full_training_config = {
    "selected_models": FULL_TRAINING_MODELS,
    "training_datasets": FULL_TRAINING_DATASETS,
    "max_epochs": FULL_MAX_EPOCHS,
    "early_stop_patience": FULL_EARLY_STOP_PATIENCE,
    "primary_metric": PRIMARY_METRIC,
    "checkpoint_policy": "save_latest_every_epoch_and_best_on_improvement",
    "resume_policy": "resume_from_latest_checkpoint_if_available",
    "full_training_root": str(FULL_TRAINING_ROOT),
}

full_training_config_json = RESULTS_05_DIR / "notebook05_full_training_config.json"

with open(full_training_config_json, "w", encoding="utf-8") as f:
    json.dump(full_training_config, f, indent=2)


def get_full_training_paths(model_name, train_dataset_name):
    run_dir = FULL_TRAINING_ROOT / model_name / f"S_{train_dataset_name}"
    checkpoint_dir = run_dir / "checkpoints"
    log_dir = run_dir / "logs"

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    return {
        "run_dir": run_dir,
        "checkpoint_dir": checkpoint_dir,
        "log_dir": log_dir,
        "latest_checkpoint": checkpoint_dir / "latest_checkpoint.pt",
        "best_checkpoint": checkpoint_dir / "best_checkpoint.pt",
        "training_log_csv": log_dir / "training_log.csv",
        "config_json": run_dir / "run_config.json",
    }


def run_full_training_loop(model_name, train_dataset_name):
    train_df = get_dataset_split(train_dataset_name, "train")
    val_df = get_dataset_split(train_dataset_name, "val")

    paths = get_full_training_paths(
        model_name=model_name,
        train_dataset_name=train_dataset_name,
    )

    config = {
        "stage": "full_training",
        "model_name": model_name,
        "segmentor_id": f"S_{train_dataset_name}",
        "train_dataset": train_dataset_name,
        "seed": SEED,
        "input_size": INPUT_SIZE,
        "batch_size": BATCH_SIZE,
        "max_epochs": FULL_MAX_EPOCHS,
        "patience": FULL_EARLY_STOP_PATIENCE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "primary_metric": PRIMARY_METRIC,
        "device": str(DEVICE),
        "use_amp": bool(USE_AMP),
    }

    with open(paths["config_json"], "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    train_loader = make_loader(train_df, shuffle=True, augment=True)
    val_loader = make_loader(val_df, shuffle=False, augment=False)

    model = build_model(model_name).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=max(3, FULL_EARLY_STOP_PATIENCE // 3),
    )

    scaler = make_grad_scaler()

    start_epoch, best_metric, history = load_checkpoint_if_available(
        paths["latest_checkpoint"],
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=DEVICE,
    )

    epochs_without_improvement = 0

    if len(history) > 0:
        recent_best = max(row["val_mean_dice_od_oc"] for row in history)
        trailing_rows = list(reversed(history))
        for row in trailing_rows:
            if row["val_mean_dice_od_oc"] >= recent_best:
                break
            epochs_without_improvement += 1

    for epoch in range(start_epoch, FULL_MAX_EPOCHS):
        epoch_start = time.time()

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
        )

        val_metrics = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=DEVICE,
        )

        current_metric = float(val_metrics[PRIMARY_METRIC])
        scheduler.step(current_metric)

        improved = current_metric > best_metric

        if improved:
            best_metric = current_metric
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        epoch_row = {
            "epoch": int(epoch),
            "model_name": model_name,
            "train_dataset": train_dataset_name,
            "train_loss": float(train_metrics["loss"]),
            "val_loss": float(val_metrics["loss"]),
            "val_dice_od": float(val_metrics["dice_od"]),
            "val_dice_oc": float(val_metrics["dice_oc"]),
            "val_mean_dice_od_oc": float(val_metrics["mean_dice_od_oc"]),
            "val_iou_od": float(val_metrics["iou_od"]),
            "val_iou_oc": float(val_metrics["iou_oc"]),
            "val_mean_iou_od_oc": float(val_metrics["mean_iou_od_oc"]),
            "best_metric": float(best_metric),
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
            "elapsed_sec": round(time.time() - epoch_start, 2),
            "improved": bool(improved),
        }

        history.append(epoch_row)
        pd.DataFrame(history).to_csv(paths["training_log_csv"], index=False)

        save_checkpoint(
            checkpoint_path=paths["latest_checkpoint"],
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch,
            best_metric=best_metric,
            config=config,
            history=history,
        )

        if improved:
            save_checkpoint(
                checkpoint_path=paths["best_checkpoint"],
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                best_metric=best_metric,
                config=config,
                history=history,
            )

        print(
            f"{model_name} | {train_dataset_name} | "
            f"epoch {epoch + 1}/{FULL_MAX_EPOCHS} | "
            f"val_mean_dice={current_metric:.4f} | "
            f"best={best_metric:.4f} | "
            f"elapsed={epoch_row['elapsed_sec']}s"
        )

        if epochs_without_improvement >= FULL_EARLY_STOP_PATIENCE:
            break

    best_row = max(history, key=lambda x: x["val_mean_dice_od_oc"])

    result = {
        "model_name": model_name,
        "train_dataset": train_dataset_name,
        "best_epoch": int(best_row["epoch"]),
        "best_val_mean_dice_od_oc": float(best_row["val_mean_dice_od_oc"]),
        "best_val_dice_od": float(best_row["val_dice_od"]),
        "best_val_dice_oc": float(best_row["val_dice_oc"]),
        "epochs_ran": len(history),
        "best_checkpoint": str(paths["best_checkpoint"]),
        "latest_checkpoint": str(paths["latest_checkpoint"]),
        "training_log": str(paths["training_log_csv"]),
    }

    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

    return result


print(f"Selected models: {FULL_TRAINING_MODELS}")
print(f"Full training runs: {len(full_training_plan)}")
print(full_training_plan[["run_id", "train_samples", "val_samples", "max_epochs", "early_stop_patience"]].to_string(index=False))
print(f"Full training plan: {full_training_plan_csv}")
print(f"Full training config: {full_training_config_json}")

if set(FULL_TRAINING_MODELS) != {"DeepLabLite", "ResidualUNetLite"}:
    raise RuntimeError(f"Unexpected full-training models: {FULL_TRAINING_MODELS}")

if len(full_training_plan) != 6:
    raise RuntimeError("Expected 6 full-training runs.")

print("Status: PASS")

In [ ]:
# Cell 12: Run one full-training job with automatic resume

RUN_ID_TO_TRAIN = "ResidualUNetLite__S_PAPILA"

run_row = full_training_plan.loc[
    full_training_plan["run_id"] == RUN_ID_TO_TRAIN
]

if len(run_row) != 1:
    raise ValueError(f"Invalid RUN_ID_TO_TRAIN: {RUN_ID_TO_TRAIN}")

run_row = run_row.iloc[0]

model_name = run_row["model_name"]
train_dataset_name = run_row["train_dataset"]

paths = get_full_training_paths(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

latest_exists = paths["latest_checkpoint"].exists()
best_exists = paths["best_checkpoint"].exists()

print(f"Run ID: {RUN_ID_TO_TRAIN}")
print(f"Model: {model_name}")
print(f"Train dataset: {train_dataset_name}")
print(f"Latest checkpoint exists: {latest_exists}")
print(f"Best checkpoint exists: {best_exists}")
print(f"Latest checkpoint: {paths['latest_checkpoint']}")
print(f"Best checkpoint: {paths['best_checkpoint']}")

result = run_full_training_loop(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

single_run_result = pd.DataFrame([result])

single_run_result_csv = (
    RESULTS_05_DIR /
    f"notebook05_full_training_result_{RUN_ID_TO_TRAIN}.csv"
)

single_run_result.to_csv(single_run_result_csv, index=False)

print("Full-training run result:")
print(single_run_result.to_string(index=False))
print(f"Run result CSV: {single_run_result_csv}")

if not Path(result["best_checkpoint"]).exists():
    raise RuntimeError("Best checkpoint was not saved.")

if not Path(result["latest_checkpoint"]).exists():
    raise RuntimeError("Latest checkpoint was not saved.")

print("Status: PASS")

In [ ]:
# Cell 13: Run next full-training job with automatic resume

RUN_ID_TO_TRAIN = "DeepLabLite__S_PAPILA"

run_row = full_training_plan.loc[
    full_training_plan["run_id"] == RUN_ID_TO_TRAIN
]

if len(run_row) != 1:
    raise ValueError(f"Invalid RUN_ID_TO_TRAIN: {RUN_ID_TO_TRAIN}")

run_row = run_row.iloc[0]

model_name = run_row["model_name"]
train_dataset_name = run_row["train_dataset"]

paths = get_full_training_paths(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

latest_exists = paths["latest_checkpoint"].exists()
best_exists = paths["best_checkpoint"].exists()

print(f"Run ID: {RUN_ID_TO_TRAIN}")
print(f"Model: {model_name}")
print(f"Train dataset: {train_dataset_name}")
print(f"Latest checkpoint exists: {latest_exists}")
print(f"Best checkpoint exists: {best_exists}")
print(f"Latest checkpoint: {paths['latest_checkpoint']}")
print(f"Best checkpoint: {paths['best_checkpoint']}")

result = run_full_training_loop(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

single_run_result = pd.DataFrame([result])

single_run_result_csv = (
    RESULTS_05_DIR /
    f"notebook05_full_training_result_{RUN_ID_TO_TRAIN}.csv"
)

single_run_result.to_csv(single_run_result_csv, index=False)

print("Full-training run result:")
print(single_run_result.to_string(index=False))
print(f"Run result CSV: {single_run_result_csv}")

if not Path(result["best_checkpoint"]).exists():
    raise RuntimeError("Best checkpoint was not saved.")

if not Path(result["latest_checkpoint"]).exists():
    raise RuntimeError("Latest checkpoint was not saved.")

print("Status: PASS")

In [ ]:
# Cell 14: Run next full-training job with automatic resume

RUN_ID_TO_TRAIN = "ResidualUNetLite__S_RIGA"

run_row = full_training_plan.loc[
    full_training_plan["run_id"] == RUN_ID_TO_TRAIN
]

if len(run_row) != 1:
    raise ValueError(f"Invalid RUN_ID_TO_TRAIN: {RUN_ID_TO_TRAIN}")

run_row = run_row.iloc[0]

model_name = run_row["model_name"]
train_dataset_name = run_row["train_dataset"]

paths = get_full_training_paths(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

latest_exists = paths["latest_checkpoint"].exists()
best_exists = paths["best_checkpoint"].exists()

print(f"Run ID: {RUN_ID_TO_TRAIN}")
print(f"Model: {model_name}")
print(f"Train dataset: {train_dataset_name}")
print(f"Latest checkpoint exists: {latest_exists}")
print(f"Best checkpoint exists: {best_exists}")
print(f"Latest checkpoint: {paths['latest_checkpoint']}")
print(f"Best checkpoint: {paths['best_checkpoint']}")

result = run_full_training_loop(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

single_run_result = pd.DataFrame([result])

single_run_result_csv = (
    RESULTS_05_DIR /
    f"notebook05_full_training_result_{RUN_ID_TO_TRAIN}.csv"
)

single_run_result.to_csv(single_run_result_csv, index=False)

print("Full-training run result:")
print(single_run_result.to_string(index=False))
print(f"Run result CSV: {single_run_result_csv}")

if not Path(result["best_checkpoint"]).exists():
    raise RuntimeError("Best checkpoint was not saved.")

if not Path(result["latest_checkpoint"]).exists():
    raise RuntimeError("Latest checkpoint was not saved.")

print("Status: PASS")

In [ ]:
# Cell 15: Run next full-training job with automatic resume

RUN_ID_TO_TRAIN = "DeepLabLite__S_RIGA"

run_row = full_training_plan.loc[
    full_training_plan["run_id"] == RUN_ID_TO_TRAIN
]

if len(run_row) != 1:
    raise ValueError(f"Invalid RUN_ID_TO_TRAIN: {RUN_ID_TO_TRAIN}")

run_row = run_row.iloc[0]

model_name = run_row["model_name"]
train_dataset_name = run_row["train_dataset"]

paths = get_full_training_paths(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

latest_exists = paths["latest_checkpoint"].exists()
best_exists = paths["best_checkpoint"].exists()

print(f"Run ID: {RUN_ID_TO_TRAIN}")
print(f"Model: {model_name}")
print(f"Train dataset: {train_dataset_name}")
print(f"Latest checkpoint exists: {latest_exists}")
print(f"Best checkpoint exists: {best_exists}")
print(f"Latest checkpoint: {paths['latest_checkpoint']}")
print(f"Best checkpoint: {paths['best_checkpoint']}")

result = run_full_training_loop(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

single_run_result = pd.DataFrame([result])

single_run_result_csv = (
    RESULTS_05_DIR /
    f"notebook05_full_training_result_{RUN_ID_TO_TRAIN}.csv"
)

single_run_result.to_csv(single_run_result_csv, index=False)

print("Full-training run result:")
print(single_run_result.to_string(index=False))
print(f"Run result CSV: {single_run_result_csv}")

if not Path(result["best_checkpoint"]).exists():
    raise RuntimeError("Best checkpoint was not saved.")

if not Path(result["latest_checkpoint"]).exists():
    raise RuntimeError("Latest checkpoint was not saved.")

print("Status: PASS")

In [ ]:
# Cell 16: Run next full-training job with automatic resume

RUN_ID_TO_TRAIN = "ResidualUNetLite__S_REFUGE"

run_row = full_training_plan.loc[
    full_training_plan["run_id"] == RUN_ID_TO_TRAIN
]

if len(run_row) != 1:
    raise ValueError(f"Invalid RUN_ID_TO_TRAIN: {RUN_ID_TO_TRAIN}")

run_row = run_row.iloc[0]

model_name = run_row["model_name"]
train_dataset_name = run_row["train_dataset"]

paths = get_full_training_paths(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

latest_exists = paths["latest_checkpoint"].exists()
best_exists = paths["best_checkpoint"].exists()

print(f"Run ID: {RUN_ID_TO_TRAIN}")
print(f"Model: {model_name}")
print(f"Train dataset: {train_dataset_name}")
print(f"Latest checkpoint exists: {latest_exists}")
print(f"Best checkpoint exists: {best_exists}")
print(f"Latest checkpoint: {paths['latest_checkpoint']}")
print(f"Best checkpoint: {paths['best_checkpoint']}")

result = run_full_training_loop(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

single_run_result = pd.DataFrame([result])

single_run_result_csv = (
    RESULTS_05_DIR /
    f"notebook05_full_training_result_{RUN_ID_TO_TRAIN}.csv"
)

single_run_result.to_csv(single_run_result_csv, index=False)

print("Full-training run result:")
print(single_run_result.to_string(index=False))
print(f"Run result CSV: {single_run_result_csv}")

if not Path(result["best_checkpoint"]).exists():
    raise RuntimeError("Best checkpoint was not saved.")

if not Path(result["latest_checkpoint"]).exists():
    raise RuntimeError("Latest checkpoint was not saved.")

print("Status: PASS")

In [ ]:
# Cell 17: Run final full-training job with automatic resume

RUN_ID_TO_TRAIN = "DeepLabLite__S_REFUGE"

run_row = full_training_plan.loc[
    full_training_plan["run_id"] == RUN_ID_TO_TRAIN
]

if len(run_row) != 1:
    raise ValueError(f"Invalid RUN_ID_TO_TRAIN: {RUN_ID_TO_TRAIN}")

run_row = run_row.iloc[0]

model_name = run_row["model_name"]
train_dataset_name = run_row["train_dataset"]

paths = get_full_training_paths(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

latest_exists = paths["latest_checkpoint"].exists()
best_exists = paths["best_checkpoint"].exists()

print(f"Run ID: {RUN_ID_TO_TRAIN}")
print(f"Model: {model_name}")
print(f"Train dataset: {train_dataset_name}")
print(f"Latest checkpoint exists: {latest_exists}")
print(f"Best checkpoint exists: {best_exists}")
print(f"Latest checkpoint: {paths['latest_checkpoint']}")
print(f"Best checkpoint: {paths['best_checkpoint']}")

result = run_full_training_loop(
    model_name=model_name,
    train_dataset_name=train_dataset_name,
)

single_run_result = pd.DataFrame([result])

single_run_result_csv = (
    RESULTS_05_DIR /
    f"notebook05_full_training_result_{RUN_ID_TO_TRAIN}.csv"
)

single_run_result.to_csv(single_run_result_csv, index=False)

print("Full-training run result:")
print(single_run_result.to_string(index=False))
print(f"Run result CSV: {single_run_result_csv}")

if not Path(result["best_checkpoint"]).exists():
    raise RuntimeError("Best checkpoint was not saved.")

if not Path(result["latest_checkpoint"]).exists():
    raise RuntimeError("Latest checkpoint was not saved.")

print("Status: PASS")

In [ ]:
# Cell 18: Consolidate full-training results

expected_full_run_ids = [
    "DeepLabLite__S_REFUGE",
    "DeepLabLite__S_PAPILA",
    "DeepLabLite__S_RIGA",
    "ResidualUNetLite__S_REFUGE",
    "ResidualUNetLite__S_PAPILA",
    "ResidualUNetLite__S_RIGA",
]

full_result_rows = []

for run_id in expected_full_run_ids:
    result_csv = RESULTS_05_DIR / f"notebook05_full_training_result_{run_id}.csv"

    if not result_csv.exists():
        raise FileNotFoundError(f"Missing full-training result CSV: {result_csv}")

    result_df = pd.read_csv(result_csv)

    if len(result_df) != 1:
        raise RuntimeError(f"Unexpected rows in result CSV {result_csv}: {len(result_df)}")

    row = result_df.iloc[0].to_dict()
    row["run_id"] = run_id

    best_checkpoint = Path(row["best_checkpoint"])
    latest_checkpoint = Path(row["latest_checkpoint"])
    training_log = Path(row["training_log"])

    row["best_checkpoint_exists"] = best_checkpoint.exists()
    row["latest_checkpoint_exists"] = latest_checkpoint.exists()
    row["training_log_exists"] = training_log.exists()

    if training_log.exists():
        log_df = pd.read_csv(training_log)
        row["log_rows"] = len(log_df)
        row["final_epoch_logged"] = int(log_df["epoch"].max())
        row["best_epoch_one_indexed"] = int(row["best_epoch"]) + 1
    else:
        row["log_rows"] = 0
        row["final_epoch_logged"] = -1
        row["best_epoch_one_indexed"] = int(row["best_epoch"]) + 1

    full_result_rows.append(row)

full_training_results = pd.DataFrame(full_result_rows)

full_training_results = full_training_results[
    [
        "run_id",
        "model_name",
        "train_dataset",
        "best_epoch",
        "best_epoch_one_indexed",
        "best_val_mean_dice_od_oc",
        "best_val_dice_od",
        "best_val_dice_oc",
        "epochs_ran",
        "log_rows",
        "best_checkpoint_exists",
        "latest_checkpoint_exists",
        "training_log_exists",
        "best_checkpoint",
        "latest_checkpoint",
        "training_log",
    ]
].sort_values(
    ["best_val_mean_dice_od_oc", "best_val_dice_oc"],
    ascending=False,
).reset_index(drop=True)

architecture_full_summary = (
    full_training_results.groupby("model_name")
    .agg(
        mean_best_val_dice=("best_val_mean_dice_od_oc", "mean"),
        mean_best_val_dice_od=("best_val_dice_od", "mean"),
        mean_best_val_dice_oc=("best_val_dice_oc", "mean"),
        min_best_val_dice=("best_val_mean_dice_od_oc", "min"),
        max_best_val_dice=("best_val_mean_dice_od_oc", "max"),
        total_epochs=("epochs_ran", "sum"),
    )
    .reset_index()
    .sort_values(["mean_best_val_dice", "min_best_val_dice"], ascending=False)
)

dataset_full_summary = (
    full_training_results.groupby("train_dataset")
    .agg(
        best_model=("model_name", lambda x: full_training_results.loc[x.index[0], "model_name"]),
        mean_best_val_dice=("best_val_mean_dice_od_oc", "mean"),
        max_best_val_dice=("best_val_mean_dice_od_oc", "max"),
    )
    .reset_index()
)

full_results_csv = RESULTS_05_DIR / "notebook05_full_training_results_consolidated.csv"
architecture_summary_csv = RESULTS_05_DIR / "notebook05_full_training_architecture_summary.csv"
dataset_summary_csv = RESULTS_05_DIR / "notebook05_full_training_dataset_summary.csv"

full_training_results.to_csv(full_results_csv, index=False)
architecture_full_summary.to_csv(architecture_summary_csv, index=False)
dataset_full_summary.to_csv(dataset_summary_csv, index=False)

missing_artifacts = int(
    (~full_training_results["best_checkpoint_exists"]).sum()
    + (~full_training_results["latest_checkpoint_exists"]).sum()
    + (~full_training_results["training_log_exists"]).sum()
)

print("Full-training results:")
print(
    full_training_results[
        [
            "run_id",
            "best_epoch_one_indexed",
            "best_val_mean_dice_od_oc",
            "best_val_dice_od",
            "best_val_dice_oc",
            "epochs_ran",
        ]
    ].to_string(index=False)
)

print("\nArchitecture summary:")
print(architecture_full_summary.to_string(index=False))

print(f"\nConsolidated results: {full_results_csv}")
print(f"Architecture summary: {architecture_summary_csv}")

if len(full_training_results) != 6:
    raise RuntimeError(f"Expected 6 full-training results, found {len(full_training_results)}")

if missing_artifacts > 0:
    raise RuntimeError(f"Missing checkpoint/log artifacts: {missing_artifacts}")

print("Status: PASS")

In [ ]:
# Cell 19: Define cross-dataset evaluation utilities

def load_best_model_for_evaluation(model_name, train_dataset_name):
    paths = get_full_training_paths(
        model_name=model_name,
        train_dataset_name=train_dataset_name,
    )

    checkpoint_path = paths["best_checkpoint"]

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing best checkpoint: {checkpoint_path}")

    model = build_model(model_name).to(DEVICE)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model, checkpoint_path


def binary_metrics_from_masks(pred_mask, true_mask, eps=1e-7):
    pred = pred_mask.astype(bool)
    true = true_mask.astype(bool)

    intersection = np.logical_and(pred, true).sum()
    union = np.logical_or(pred, true).sum()

    pred_area = pred.sum()
    true_area = true.sum()

    dice = (2.0 * intersection + eps) / (pred_area + true_area + eps)
    iou = (intersection + eps) / (union + eps)

    if pred_area > 0 and true_area > 0:
        pred_yx = np.column_stack(np.where(pred))
        true_yx = np.column_stack(np.where(true))

        pred_centroid = pred_yx.mean(axis=0)
        true_centroid = true_yx.mean(axis=0)

        centroid_error_px = float(np.linalg.norm(pred_centroid - true_centroid))
    else:
        centroid_error_px = np.nan

    area_error_rel = float(abs(pred_area - true_area) / max(true_area, 1))

    return {
        "dice": float(dice),
        "iou": float(iou),
        "centroid_error_px": centroid_error_px,
        "area_error_rel": area_error_rel,
        "pred_area": int(pred_area),
        "true_area": int(true_area),
        "pred_empty": bool(pred_area == 0),
        "true_empty": bool(true_area == 0),
    }


@torch.no_grad()
def evaluate_model_on_dataset(model_name, train_dataset_name, test_dataset_name):
    model, checkpoint_path = load_best_model_for_evaluation(
        model_name=model_name,
        train_dataset_name=train_dataset_name,
    )

    test_df = get_dataset_split(test_dataset_name, "test")
    test_loader = make_loader(test_df, shuffle=False, augment=False)

    sample_rows = []

    for batch in test_loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].cpu().numpy()
        sample_ids = batch["sample_id"]

        with amp_autocast_context():
            logits = model(images)

        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()

        for i, sample_id in enumerate(sample_ids):
            true_mask = masks[i]
            pred_mask = preds[i]

            od_metrics = binary_metrics_from_masks(
                pred_mask=(pred_mask == 1),
                true_mask=(true_mask == 1),
            )

            oc_metrics = binary_metrics_from_masks(
                pred_mask=(pred_mask == 2),
                true_mask=(true_mask == 2),
            )

            sample_rows.append(
                {
                    "model_name": model_name,
                    "segmentor_id": f"S_{train_dataset_name}",
                    "train_dataset": train_dataset_name,
                    "test_dataset": test_dataset_name,
                    "sample_id": sample_id,
                    "evaluation_type": "internal" if train_dataset_name == test_dataset_name else "cross_dataset_external",
                    "dice_od": od_metrics["dice"],
                    "dice_oc": oc_metrics["dice"],
                    "iou_od": od_metrics["iou"],
                    "iou_oc": oc_metrics["iou"],
                    "mean_dice_od_oc": float(0.5 * (od_metrics["dice"] + oc_metrics["dice"])),
                    "mean_iou_od_oc": float(0.5 * (od_metrics["iou"] + oc_metrics["iou"])),
                    "centroid_error_od_px": od_metrics["centroid_error_px"],
                    "centroid_error_oc_px": oc_metrics["centroid_error_px"],
                    "area_error_od_rel": od_metrics["area_error_rel"],
                    "area_error_oc_rel": oc_metrics["area_error_rel"],
                    "pred_empty_od": od_metrics["pred_empty"],
                    "pred_empty_oc": oc_metrics["pred_empty"],
                    "checkpoint_path": str(checkpoint_path),
                }
            )

    del model
    torch.cuda.empty_cache()

    sample_metrics = pd.DataFrame(sample_rows)

    summary = {
        "model_name": model_name,
        "segmentor_id": f"S_{train_dataset_name}",
        "train_dataset": train_dataset_name,
        "test_dataset": test_dataset_name,
        "evaluation_type": "internal" if train_dataset_name == test_dataset_name else "cross_dataset_external",
        "n_samples": int(len(sample_metrics)),
        "dice_od_mean": float(sample_metrics["dice_od"].mean()),
        "dice_oc_mean": float(sample_metrics["dice_oc"].mean()),
        "mean_dice_od_oc": float(sample_metrics["mean_dice_od_oc"].mean()),
        "iou_od_mean": float(sample_metrics["iou_od"].mean()),
        "iou_oc_mean": float(sample_metrics["iou_oc"].mean()),
        "mean_iou_od_oc": float(sample_metrics["mean_iou_od_oc"].mean()),
        "centroid_error_od_px_mean": float(sample_metrics["centroid_error_od_px"].mean()),
        "centroid_error_oc_px_mean": float(sample_metrics["centroid_error_oc_px"].mean()),
        "area_error_od_rel_mean": float(sample_metrics["area_error_od_rel"].mean()),
        "area_error_oc_rel_mean": float(sample_metrics["area_error_oc_rel"].mean()),
        "pred_empty_od": int(sample_metrics["pred_empty_od"].sum()),
        "pred_empty_oc": int(sample_metrics["pred_empty_oc"].sum()),
        "checkpoint_path": str(checkpoint_path),
    }

    return sample_metrics, summary


evaluation_utility_check = {
    "selected_models": FULL_TRAINING_MODELS,
    "train_datasets": FULL_TRAINING_DATASETS,
    "expected_full_segmentors": len(FULL_TRAINING_MODELS) * len(FULL_TRAINING_DATASETS),
    "expected_cross_dataset_evaluations": len(FULL_TRAINING_MODELS) * len(FULL_TRAINING_DATASETS) * len(FULL_TRAINING_DATASETS),
}

evaluation_utility_json = RESULTS_05_DIR / "notebook05_evaluation_utility_config.json"

with open(evaluation_utility_json, "w", encoding="utf-8") as f:
    json.dump(evaluation_utility_check, f, indent=2)

print(f"Selected models: {FULL_TRAINING_MODELS}")
print(f"Train datasets: {FULL_TRAINING_DATASETS}")
print(f"Expected segmentors: {evaluation_utility_check['expected_full_segmentors']}")
print(f"Expected evaluations: {evaluation_utility_check['expected_cross_dataset_evaluations']}")
print(f"Evaluation utility config: {evaluation_utility_json}")
print("Status: PASS")

In [ ]:
# Cell 20: Run internal and cross-dataset evaluation for all full-trained segmentors

evaluation_sample_dir = RESULTS_05_DIR / "cross_dataset_sample_metrics"
evaluation_sample_dir.mkdir(parents=True, exist_ok=True)

evaluation_summary_rows = []

for model_name in FULL_TRAINING_MODELS:
    for train_dataset_name in FULL_TRAINING_DATASETS:
        for test_dataset_name in FULL_TRAINING_DATASETS:
            pair_id = f"{model_name}__train_{train_dataset_name}__test_{test_dataset_name}"

            sample_csv = evaluation_sample_dir / f"{pair_id}_sample_metrics.csv"

            if sample_csv.exists():
                sample_metrics = pd.read_csv(sample_csv)

                summary = {
                    "model_name": model_name,
                    "segmentor_id": f"S_{train_dataset_name}",
                    "train_dataset": train_dataset_name,
                    "test_dataset": test_dataset_name,
                    "evaluation_type": "internal" if train_dataset_name == test_dataset_name else "cross_dataset_external",
                    "n_samples": int(len(sample_metrics)),
                    "dice_od_mean": float(sample_metrics["dice_od"].mean()),
                    "dice_oc_mean": float(sample_metrics["dice_oc"].mean()),
                    "mean_dice_od_oc": float(sample_metrics["mean_dice_od_oc"].mean()),
                    "iou_od_mean": float(sample_metrics["iou_od"].mean()),
                    "iou_oc_mean": float(sample_metrics["iou_oc"].mean()),
                    "mean_iou_od_oc": float(sample_metrics["mean_iou_od_oc"].mean()),
                    "centroid_error_od_px_mean": float(sample_metrics["centroid_error_od_px"].mean()),
                    "centroid_error_oc_px_mean": float(sample_metrics["centroid_error_oc_px"].mean()),
                    "area_error_od_rel_mean": float(sample_metrics["area_error_od_rel"].mean()),
                    "area_error_oc_rel_mean": float(sample_metrics["area_error_oc_rel"].mean()),
                    "pred_empty_od": int(sample_metrics["pred_empty_od"].sum()),
                    "pred_empty_oc": int(sample_metrics["pred_empty_oc"].sum()),
                    "checkpoint_path": sample_metrics["checkpoint_path"].iloc[0],
                    "status": "loaded_existing",
                }
            else:
                sample_metrics, summary = evaluate_model_on_dataset(
                    model_name=model_name,
                    train_dataset_name=train_dataset_name,
                    test_dataset_name=test_dataset_name,
                )

                sample_metrics.to_csv(sample_csv, index=False)
                summary["status"] = "evaluated"

            evaluation_summary_rows.append(summary)

            live_summary = pd.DataFrame(evaluation_summary_rows)
            live_summary.to_csv(
                RESULTS_05_DIR / "notebook05_cross_dataset_evaluation_summary_live.csv",
                index=False,
            )

            print(
                f"{model_name} | train={train_dataset_name} | test={test_dataset_name} | "
                f"mean_dice={summary['mean_dice_od_oc']:.4f} | "
                f"OD={summary['dice_od_mean']:.4f} | "
                f"OC={summary['dice_oc_mean']:.4f} | "
                f"{summary['status']}"
            )

cross_dataset_evaluation_summary = pd.DataFrame(evaluation_summary_rows)

cross_dataset_evaluation_summary_csv = (
    RESULTS_05_DIR / "notebook05_cross_dataset_evaluation_summary.csv"
)

cross_dataset_evaluation_summary.to_csv(cross_dataset_evaluation_summary_csv, index=False)

architecture_cross_dataset_summary = (
    cross_dataset_evaluation_summary.groupby(["model_name", "evaluation_type"])
    .agg(
        n_pairs=("mean_dice_od_oc", "count"),
        mean_dice=("mean_dice_od_oc", "mean"),
        mean_dice_od=("dice_od_mean", "mean"),
        mean_dice_oc=("dice_oc_mean", "mean"),
        min_pair_dice=("mean_dice_od_oc", "min"),
        max_pair_dice=("mean_dice_od_oc", "max"),
        total_pred_empty_od=("pred_empty_od", "sum"),
        total_pred_empty_oc=("pred_empty_oc", "sum"),
    )
    .reset_index()
)

architecture_cross_dataset_summary_csv = (
    RESULTS_05_DIR / "notebook05_architecture_cross_dataset_summary.csv"
)

architecture_cross_dataset_summary.to_csv(
    architecture_cross_dataset_summary_csv,
    index=False,
)

print("\nCross-dataset evaluation summary:")
print(
    cross_dataset_evaluation_summary[
        [
            "model_name",
            "train_dataset",
            "test_dataset",
            "evaluation_type",
            "n_samples",
            "mean_dice_od_oc",
            "dice_od_mean",
            "dice_oc_mean",
            "pred_empty_od",
            "pred_empty_oc",
        ]
    ].to_string(index=False)
)

print("\nArchitecture-level summary:")
print(architecture_cross_dataset_summary.to_string(index=False))

print(f"\nEvaluation summary: {cross_dataset_evaluation_summary_csv}")

if len(cross_dataset_evaluation_summary) != 18:
    raise RuntimeError(f"Expected 18 evaluation pairs, found {len(cross_dataset_evaluation_summary)}")

print("Status: PASS")

In [ ]:
# Cell 21: Analyze cross-dataset results and select final candidate strategy

EVAL_SUMMARY_CSV = RESULTS_05_DIR / "notebook05_cross_dataset_evaluation_summary.csv"

if not EVAL_SUMMARY_CSV.exists():
    raise FileNotFoundError(f"Missing evaluation summary: {EVAL_SUMMARY_CSV}")

cross_dataset_evaluation_summary = pd.read_csv(EVAL_SUMMARY_CSV)

internal_results = cross_dataset_evaluation_summary[
    cross_dataset_evaluation_summary["evaluation_type"] == "internal"
].copy()

external_results = cross_dataset_evaluation_summary[
    cross_dataset_evaluation_summary["evaluation_type"] == "cross_dataset_external"
].copy()

architecture_decision_summary = (
    cross_dataset_evaluation_summary.groupby("model_name")
    .agg(
        all_pairs_mean_dice=("mean_dice_od_oc", "mean"),
        internal_mean_dice=("mean_dice_od_oc", lambda x: np.nan),
    )
    .reset_index()
)

internal_arch = (
    internal_results.groupby("model_name")
    .agg(
        internal_mean_dice=("mean_dice_od_oc", "mean"),
        internal_min_dice=("mean_dice_od_oc", "min"),
        internal_pred_empty_od=("pred_empty_od", "sum"),
        internal_pred_empty_oc=("pred_empty_oc", "sum"),
    )
    .reset_index()
)

external_arch = (
    external_results.groupby("model_name")
    .agg(
        external_mean_dice=("mean_dice_od_oc", "mean"),
        external_min_dice=("mean_dice_od_oc", "min"),
        external_max_dice=("mean_dice_od_oc", "max"),
        external_pred_empty_od=("pred_empty_od", "sum"),
        external_pred_empty_oc=("pred_empty_oc", "sum"),
    )
    .reset_index()
)

architecture_decision_summary = internal_arch.merge(
    external_arch,
    on="model_name",
    how="inner",
)

architecture_decision_summary["external_empty_predictions"] = (
    architecture_decision_summary["external_pred_empty_od"] +
    architecture_decision_summary["external_pred_empty_oc"]
)

architecture_decision_summary["decision_score"] = (
    architecture_decision_summary["internal_mean_dice"]
    + architecture_decision_summary["external_mean_dice"]
    - 0.001 * architecture_decision_summary["external_empty_predictions"]
)

architecture_decision_summary = architecture_decision_summary.sort_values(
    ["decision_score", "external_mean_dice", "internal_mean_dice"],
    ascending=False,
).reset_index(drop=True)

domain_shift_matrix = (
    cross_dataset_evaluation_summary.pivot_table(
        index=["model_name", "train_dataset"],
        columns="test_dataset",
        values="mean_dice_od_oc",
        aggfunc="mean",
    )
    .reset_index()
)

test_domain_difficulty = (
    external_results.groupby(["model_name", "test_dataset"])
    .agg(
        external_test_mean_dice=("mean_dice_od_oc", "mean"),
        external_test_min_dice=("mean_dice_od_oc", "min"),
        pred_empty_od=("pred_empty_od", "sum"),
        pred_empty_oc=("pred_empty_oc", "sum"),
    )
    .reset_index()
    .sort_values(["model_name", "external_test_mean_dice"])
)

selected_primary_architecture = architecture_decision_summary.iloc[0]["model_name"]
selected_secondary_architecture = architecture_decision_summary.iloc[1]["model_name"]

final_strategy = {
    "stage": "post_cross_dataset_decision",
    "primary_architecture": selected_primary_architecture,
    "secondary_architecture": selected_secondary_architecture,
    "decision_basis": [
        "internal validation Dice",
        "cross-dataset external Dice",
        "external prediction failure count",
        "domain-shift behavior across REFUGE, PAPILA, and RIGA",
    ],
    "key_interpretation": (
        "Single-source segmentors show strong internal performance but limited external "
        "generalization to PAPILA. A final multi-source segmentor is justified after "
        "dataset-specific cross-domain validation."
    ),
    "next_step": "train multi-source final segmentors using REFUGE, PAPILA, and RIGA training splits.",
}

architecture_decision_csv = RESULTS_05_DIR / "notebook05_architecture_decision_summary.csv"
domain_shift_matrix_csv = RESULTS_05_DIR / "notebook05_domain_shift_matrix.csv"
test_domain_difficulty_csv = RESULTS_05_DIR / "notebook05_test_domain_difficulty.csv"
final_strategy_json = RESULTS_05_DIR / "notebook05_final_segmentor_strategy.json"

architecture_decision_summary.to_csv(architecture_decision_csv, index=False)
domain_shift_matrix.to_csv(domain_shift_matrix_csv, index=False)
test_domain_difficulty.to_csv(test_domain_difficulty_csv, index=False)

with open(final_strategy_json, "w", encoding="utf-8") as f:
    json.dump(final_strategy, f, indent=2)

print("Architecture decision summary:")
print(architecture_decision_summary.to_string(index=False))

print("\nDomain-shift matrix:")
print(domain_shift_matrix.to_string(index=False))

print("\nExternal test-domain difficulty:")
print(test_domain_difficulty.to_string(index=False))

print(f"\nArchitecture decision: {architecture_decision_csv}")
print(f"Domain-shift matrix: {domain_shift_matrix_csv}")
print(f"Final strategy: {final_strategy_json}")

if selected_primary_architecture not in FULL_TRAINING_MODELS:
    raise RuntimeError(f"Unexpected primary architecture: {selected_primary_architecture}")

if len(domain_shift_matrix) != 6:
    raise RuntimeError("Unexpected domain-shift matrix size.")

print("Status: PASS")

In [ ]:
# Cell 22: Configure multi-source final training plan

from torch.utils.data import WeightedRandomSampler

MULTISOURCE_MODELS = [
    "DeepLabLite",
    "ResidualUNetLite",
]

MULTISOURCE_TRAIN_DATASETS = ["REFUGE", "PAPILA", "RIGA"]

MULTISOURCE_MAX_EPOCHS = 100
MULTISOURCE_EARLY_STOP_PATIENCE = 18

MULTISOURCE_ROOT = MODEL_ROOT / "multisource_final"
MULTISOURCE_ROOT.mkdir(parents=True, exist_ok=True)

multisource_train_df = segmentation_manifest[
    segmentation_manifest["segmentation_split"] == "train"
].copy().reset_index(drop=True)

multisource_val_df = segmentation_manifest[
    segmentation_manifest["segmentation_split"] == "val"
].copy().reset_index(drop=True)

multisource_test_df = segmentation_manifest[
    segmentation_manifest["segmentation_split"] == "test"
].copy().reset_index(drop=True)

def make_balanced_multisource_loader(dataframe, batch_size=BATCH_SIZE, augment=True):
    df = dataframe.reset_index(drop=True).copy()
    dataset_counts = df["dataset"].value_counts().to_dict()
    sample_weights = df["dataset"].map(lambda name: 1.0 / dataset_counts[name]).astype(float).values

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(df),
        replacement=True,
    )

    dataset = ODOCSegmentationDataset(
        dataframe=df,
        augment=augment,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
        drop_last=False,
    )


def get_multisource_paths(model_name):
    run_dir = MULTISOURCE_ROOT / model_name
    checkpoint_dir = run_dir / "checkpoints"
    log_dir = run_dir / "logs"

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    return {
        "run_dir": run_dir,
        "checkpoint_dir": checkpoint_dir,
        "log_dir": log_dir,
        "latest_checkpoint": checkpoint_dir / "latest_checkpoint.pt",
        "best_checkpoint": checkpoint_dir / "best_checkpoint.pt",
        "training_log_csv": log_dir / "training_log.csv",
        "config_json": run_dir / "run_config.json",
    }


def validate_by_dataset(model, dataframe, criterion, device):
    rows = []

    for dataset_name in MULTISOURCE_TRAIN_DATASETS:
        dataset_df = dataframe[dataframe["dataset"] == dataset_name].copy()

        if len(dataset_df) == 0:
            continue

        loader = make_loader(dataset_df, shuffle=False, augment=False)
        metrics = validate_one_epoch(
            model=model,
            loader=loader,
            criterion=criterion,
            device=device,
        )

        rows.append(
            {
                "dataset": dataset_name,
                "loss": float(metrics["loss"]),
                "dice_od": float(metrics["dice_od"]),
                "dice_oc": float(metrics["dice_oc"]),
                "mean_dice_od_oc": float(metrics["mean_dice_od_oc"]),
                "iou_od": float(metrics["iou_od"]),
                "iou_oc": float(metrics["iou_oc"]),
                "mean_iou_od_oc": float(metrics["mean_iou_od_oc"]),
            }
        )

    return pd.DataFrame(rows)


def run_multisource_training_loop(model_name):
    paths = get_multisource_paths(model_name)

    config = {
        "stage": "multisource_final_training",
        "model_name": model_name,
        "train_datasets": MULTISOURCE_TRAIN_DATASETS,
        "seed": SEED,
        "input_size": INPUT_SIZE,
        "batch_size": BATCH_SIZE,
        "max_epochs": MULTISOURCE_MAX_EPOCHS,
        "patience": MULTISOURCE_EARLY_STOP_PATIENCE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "primary_metric": PRIMARY_METRIC,
        "sampling": "dataset_balanced_weighted_sampler",
        "device": str(DEVICE),
        "use_amp": bool(USE_AMP),
    }

    with open(paths["config_json"], "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    train_loader = make_balanced_multisource_loader(
        multisource_train_df,
        batch_size=BATCH_SIZE,
        augment=True,
    )

    val_loader = make_loader(
        multisource_val_df,
        shuffle=False,
        augment=False,
    )

    model = build_model(model_name).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=max(3, MULTISOURCE_EARLY_STOP_PATIENCE // 3),
    )

    scaler = make_grad_scaler()

    start_epoch, best_metric, history = load_checkpoint_if_available(
        paths["latest_checkpoint"],
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=DEVICE,
    )

    epochs_without_improvement = 0

    for epoch in range(start_epoch, MULTISOURCE_MAX_EPOCHS):
        epoch_start = time.time()

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=DEVICE,
        )

        val_metrics = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=DEVICE,
        )

        val_by_dataset = validate_by_dataset(
            model=model,
            dataframe=multisource_val_df,
            criterion=criterion,
            device=DEVICE,
        )

        current_metric = float(val_metrics[PRIMARY_METRIC])
        scheduler.step(current_metric)

        improved = current_metric > best_metric

        if improved:
            best_metric = current_metric
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        dataset_metric_prefix = {}

        for _, row in val_by_dataset.iterrows():
            dataset_key = row["dataset"].lower()
            dataset_metric_prefix[f"val_{dataset_key}_mean_dice"] = float(row["mean_dice_od_oc"])
            dataset_metric_prefix[f"val_{dataset_key}_dice_od"] = float(row["dice_od"])
            dataset_metric_prefix[f"val_{dataset_key}_dice_oc"] = float(row["dice_oc"])

        epoch_row = {
            "epoch": int(epoch),
            "model_name": model_name,
            "train_dataset": "MULTISOURCE",
            "train_loss": float(train_metrics["loss"]),
            "val_loss": float(val_metrics["loss"]),
            "val_dice_od": float(val_metrics["dice_od"]),
            "val_dice_oc": float(val_metrics["dice_oc"]),
            "val_mean_dice_od_oc": float(val_metrics["mean_dice_od_oc"]),
            "val_iou_od": float(val_metrics["iou_od"]),
            "val_iou_oc": float(val_metrics["iou_oc"]),
            "val_mean_iou_od_oc": float(val_metrics["mean_iou_od_oc"]),
            "best_metric": float(best_metric),
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
            "elapsed_sec": round(time.time() - epoch_start, 2),
            "improved": bool(improved),
            **dataset_metric_prefix,
        }

        history.append(epoch_row)
        pd.DataFrame(history).to_csv(paths["training_log_csv"], index=False)

        save_checkpoint(
            checkpoint_path=paths["latest_checkpoint"],
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch,
            best_metric=best_metric,
            config=config,
            history=history,
        )

        if improved:
            save_checkpoint(
                checkpoint_path=paths["best_checkpoint"],
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                best_metric=best_metric,
                config=config,
                history=history,
            )

        print(
            f"{model_name} | MULTISOURCE | "
            f"epoch {epoch + 1}/{MULTISOURCE_MAX_EPOCHS} | "
            f"val_mean_dice={current_metric:.4f} | "
            f"best={best_metric:.4f} | "
            f"elapsed={epoch_row['elapsed_sec']}s"
        )

        if epochs_without_improvement >= MULTISOURCE_EARLY_STOP_PATIENCE:
            break

    best_row = max(history, key=lambda x: x["val_mean_dice_od_oc"])

    result = {
        "model_name": model_name,
        "train_dataset": "MULTISOURCE",
        "best_epoch": int(best_row["epoch"]),
        "best_epoch_one_indexed": int(best_row["epoch"]) + 1,
        "best_val_mean_dice_od_oc": float(best_row["val_mean_dice_od_oc"]),
        "best_val_dice_od": float(best_row["val_dice_od"]),
        "best_val_dice_oc": float(best_row["val_dice_oc"]),
        "epochs_ran": len(history),
        "best_checkpoint": str(paths["best_checkpoint"]),
        "latest_checkpoint": str(paths["latest_checkpoint"]),
        "training_log": str(paths["training_log_csv"]),
    }

    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

    return result


multisource_plan = pd.DataFrame(
    [
        {
            "run_id": f"{model_name}__S_MULTISOURCE",
            "model_name": model_name,
            "train_dataset": "MULTISOURCE",
            "train_samples": len(multisource_train_df),
            "val_samples": len(multisource_val_df),
            "test_samples": len(multisource_test_df),
            "max_epochs": MULTISOURCE_MAX_EPOCHS,
            "early_stop_patience": MULTISOURCE_EARLY_STOP_PATIENCE,
            "run_dir": str(get_multisource_paths(model_name)["run_dir"]),
        }
        for model_name in MULTISOURCE_MODELS
    ]
)

multisource_plan_csv = RESULTS_05_DIR / "notebook05_multisource_training_plan.csv"
multisource_plan.to_csv(multisource_plan_csv, index=False)

print(f"Multisource train samples: {len(multisource_train_df)}")
print(f"Multisource val samples: {len(multisource_val_df)}")
print(f"Multisource test samples: {len(multisource_test_df)}")
print(multisource_train_df.groupby("dataset").size().to_string())
print(f"Multisource training plan: {multisource_plan_csv}")

if len(multisource_train_df) != 1261:
    raise RuntimeError(f"Unexpected multisource train size: {len(multisource_train_df)}")

if len(multisource_val_df) != 586:
    raise RuntimeError(f"Unexpected multisource val size: {len(multisource_val_df)}")

if len(multisource_plan) != 2:
    raise RuntimeError("Expected 2 multisource model runs.")

print("Status: PASS")

In [ ]:
# Cell 23: Run one multisource final-training job with automatic resume

MULTISOURCE_RUN_MODEL = "ResidualUNetLite"

if MULTISOURCE_RUN_MODEL not in MULTISOURCE_MODELS:
    raise ValueError(f"Invalid MULTISOURCE_RUN_MODEL: {MULTISOURCE_RUN_MODEL}")

paths = get_multisource_paths(MULTISOURCE_RUN_MODEL)

latest_exists = paths["latest_checkpoint"].exists()
best_exists = paths["best_checkpoint"].exists()

print(f"Run ID: {MULTISOURCE_RUN_MODEL}__S_MULTISOURCE")
print(f"Model: {MULTISOURCE_RUN_MODEL}")
print("Train dataset: MULTISOURCE")
print(f"Latest checkpoint exists: {latest_exists}")
print(f"Best checkpoint exists: {best_exists}")
print(f"Latest checkpoint: {paths['latest_checkpoint']}")
print(f"Best checkpoint: {paths['best_checkpoint']}")

result = run_multisource_training_loop(
    model_name=MULTISOURCE_RUN_MODEL,
)

multisource_single_result = pd.DataFrame([result])

multisource_single_result_csv = (
    RESULTS_05_DIR /
    f"notebook05_multisource_training_result_{MULTISOURCE_RUN_MODEL}.csv"
)

multisource_single_result.to_csv(multisource_single_result_csv, index=False)

print("Multisource training result:")
print(multisource_single_result.to_string(index=False))
print(f"Run result CSV: {multisource_single_result_csv}")

if not Path(result["best_checkpoint"]).exists():
    raise RuntimeError("Best checkpoint was not saved.")

if not Path(result["latest_checkpoint"]).exists():
    raise RuntimeError("Latest checkpoint was not saved.")

print("Status: PASS")

In [ ]:
# Cell 24: Run second multisource final-training job with automatic resume

MULTISOURCE_RUN_MODEL = "DeepLabLite"

if MULTISOURCE_RUN_MODEL not in MULTISOURCE_MODELS:
    raise ValueError(f"Invalid MULTISOURCE_RUN_MODEL: {MULTISOURCE_RUN_MODEL}")

paths = get_multisource_paths(MULTISOURCE_RUN_MODEL)

latest_exists = paths["latest_checkpoint"].exists()
best_exists = paths["best_checkpoint"].exists()

print(f"Run ID: {MULTISOURCE_RUN_MODEL}__S_MULTISOURCE")
print(f"Model: {MULTISOURCE_RUN_MODEL}")
print("Train dataset: MULTISOURCE")
print(f"Latest checkpoint exists: {latest_exists}")
print(f"Best checkpoint exists: {best_exists}")
print(f"Latest checkpoint: {paths['latest_checkpoint']}")
print(f"Best checkpoint: {paths['best_checkpoint']}")

result = run_multisource_training_loop(
    model_name=MULTISOURCE_RUN_MODEL,
)

multisource_single_result = pd.DataFrame([result])

multisource_single_result_csv = (
    RESULTS_05_DIR /
    f"notebook05_multisource_training_result_{MULTISOURCE_RUN_MODEL}.csv"
)

multisource_single_result.to_csv(multisource_single_result_csv, index=False)

print("Multisource training result:")
print(multisource_single_result.to_string(index=False))
print(f"Run result CSV: {multisource_single_result_csv}")

if not Path(result["best_checkpoint"]).exists():
    raise RuntimeError("Best checkpoint was not saved.")

if not Path(result["latest_checkpoint"]).exists():
    raise RuntimeError("Latest checkpoint was not saved.")

print("Status: PASS")

In [ ]:
# Cell 25: Consolidate multisource training results

expected_multisource_models = [
    "DeepLabLite",
    "ResidualUNetLite",
]

multisource_result_rows = []

for model_name in expected_multisource_models:
    result_csv = RESULTS_05_DIR / f"notebook05_multisource_training_result_{model_name}.csv"

    if not result_csv.exists():
        raise FileNotFoundError(f"Missing multisource result CSV: {result_csv}")

    result_df = pd.read_csv(result_csv)

    if len(result_df) != 1:
        raise RuntimeError(f"Unexpected rows in {result_csv}: {len(result_df)}")

    row = result_df.iloc[0].to_dict()
    row["run_id"] = f"{model_name}__S_MULTISOURCE"

    best_checkpoint = Path(row["best_checkpoint"])
    latest_checkpoint = Path(row["latest_checkpoint"])
    training_log = Path(row["training_log"])

    row["best_checkpoint_exists"] = best_checkpoint.exists()
    row["latest_checkpoint_exists"] = latest_checkpoint.exists()
    row["training_log_exists"] = training_log.exists()

    if training_log.exists():
        log_df = pd.read_csv(training_log)
        row["log_rows"] = len(log_df)
        row["final_epoch_logged"] = int(log_df["epoch"].max())
    else:
        row["log_rows"] = 0
        row["final_epoch_logged"] = -1

    multisource_result_rows.append(row)

multisource_training_results = pd.DataFrame(multisource_result_rows)

multisource_training_results = multisource_training_results[
    [
        "run_id",
        "model_name",
        "train_dataset",
        "best_epoch",
        "best_epoch_one_indexed",
        "best_val_mean_dice_od_oc",
        "best_val_dice_od",
        "best_val_dice_oc",
        "epochs_ran",
        "log_rows",
        "best_checkpoint_exists",
        "latest_checkpoint_exists",
        "training_log_exists",
        "best_checkpoint",
        "latest_checkpoint",
        "training_log",
    ]
].sort_values(
    ["best_val_mean_dice_od_oc", "best_val_dice_oc"],
    ascending=False,
).reset_index(drop=True)

multisource_training_results_csv = (
    RESULTS_05_DIR / "notebook05_multisource_training_results_consolidated.csv"
)

multisource_training_results.to_csv(multisource_training_results_csv, index=False)

missing_artifacts = int(
    (~multisource_training_results["best_checkpoint_exists"]).sum()
    + (~multisource_training_results["latest_checkpoint_exists"]).sum()
    + (~multisource_training_results["training_log_exists"]).sum()
)

print("Multisource training results:")
print(
    multisource_training_results[
        [
            "run_id",
            "best_epoch_one_indexed",
            "best_val_mean_dice_od_oc",
            "best_val_dice_od",
            "best_val_dice_oc",
            "epochs_ran",
        ]
    ].to_string(index=False)
)

print(f"Consolidated multisource results: {multisource_training_results_csv}")

if len(multisource_training_results) != 2:
    raise RuntimeError(f"Expected 2 multisource results, found {len(multisource_training_results)}")

if missing_artifacts > 0:
    raise RuntimeError(f"Missing multisource checkpoint/log artifacts: {missing_artifacts}")

print("Status: PASS")

In [ ]:
# Cell 26: Evaluate multisource final models on all test sets

def load_multisource_model_for_evaluation(model_name):
    paths = get_multisource_paths(model_name)
    checkpoint_path = paths["best_checkpoint"]

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing multisource best checkpoint: {checkpoint_path}")

    model = build_model(model_name).to(DEVICE)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model, checkpoint_path


@torch.no_grad()
def evaluate_multisource_model_on_dataset(model_name, test_dataset_name):
    model, checkpoint_path = load_multisource_model_for_evaluation(model_name)

    test_df = get_dataset_split(test_dataset_name, "test")
    test_loader = make_loader(test_df, shuffle=False, augment=False)

    sample_rows = []

    for batch in test_loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].cpu().numpy()
        sample_ids = batch["sample_id"]

        with amp_autocast_context():
            logits = model(images)

        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()

        for i, sample_id in enumerate(sample_ids):
            true_mask = masks[i]
            pred_mask = preds[i]

            od_metrics = binary_metrics_from_masks(
                pred_mask=(pred_mask == 1),
                true_mask=(true_mask == 1),
            )

            oc_metrics = binary_metrics_from_masks(
                pred_mask=(pred_mask == 2),
                true_mask=(true_mask == 2),
            )

            sample_rows.append(
                {
                    "model_name": model_name,
                    "segmentor_id": "S_MULTISOURCE",
                    "train_dataset": "MULTISOURCE",
                    "test_dataset": test_dataset_name,
                    "sample_id": sample_id,
                    "dice_od": od_metrics["dice"],
                    "dice_oc": oc_metrics["dice"],
                    "iou_od": od_metrics["iou"],
                    "iou_oc": oc_metrics["iou"],
                    "mean_dice_od_oc": float(0.5 * (od_metrics["dice"] + oc_metrics["dice"])),
                    "mean_iou_od_oc": float(0.5 * (od_metrics["iou"] + oc_metrics["iou"])),
                    "centroid_error_od_px": od_metrics["centroid_error_px"],
                    "centroid_error_oc_px": oc_metrics["centroid_error_px"],
                    "area_error_od_rel": od_metrics["area_error_rel"],
                    "area_error_oc_rel": oc_metrics["area_error_rel"],
                    "pred_empty_od": od_metrics["pred_empty"],
                    "pred_empty_oc": oc_metrics["pred_empty"],
                    "checkpoint_path": str(checkpoint_path),
                }
            )

    del model
    torch.cuda.empty_cache()

    sample_metrics = pd.DataFrame(sample_rows)

    summary = {
        "model_name": model_name,
        "segmentor_id": "S_MULTISOURCE",
        "train_dataset": "MULTISOURCE",
        "test_dataset": test_dataset_name,
        "n_samples": int(len(sample_metrics)),
        "dice_od_mean": float(sample_metrics["dice_od"].mean()),
        "dice_oc_mean": float(sample_metrics["dice_oc"].mean()),
        "mean_dice_od_oc": float(sample_metrics["mean_dice_od_oc"].mean()),
        "iou_od_mean": float(sample_metrics["iou_od"].mean()),
        "iou_oc_mean": float(sample_metrics["iou_oc"].mean()),
        "mean_iou_od_oc": float(sample_metrics["mean_iou_od_oc"].mean()),
        "centroid_error_od_px_mean": float(sample_metrics["centroid_error_od_px"].mean()),
        "centroid_error_oc_px_mean": float(sample_metrics["centroid_error_oc_px"].mean()),
        "area_error_od_rel_mean": float(sample_metrics["area_error_od_rel"].mean()),
        "area_error_oc_rel_mean": float(sample_metrics["area_error_oc_rel"].mean()),
        "pred_empty_od": int(sample_metrics["pred_empty_od"].sum()),
        "pred_empty_oc": int(sample_metrics["pred_empty_oc"].sum()),
        "checkpoint_path": str(checkpoint_path),
    }

    return sample_metrics, summary


multisource_eval_sample_dir = RESULTS_05_DIR / "multisource_sample_metrics"
multisource_eval_sample_dir.mkdir(parents=True, exist_ok=True)

multisource_eval_rows = []

for model_name in MULTISOURCE_MODELS:
    for test_dataset_name in FULL_TRAINING_DATASETS:
        pair_id = f"{model_name}__train_MULTISOURCE__test_{test_dataset_name}"
        sample_csv = multisource_eval_sample_dir / f"{pair_id}_sample_metrics.csv"

        if sample_csv.exists():
            sample_metrics = pd.read_csv(sample_csv)

            summary = {
                "model_name": model_name,
                "segmentor_id": "S_MULTISOURCE",
                "train_dataset": "MULTISOURCE",
                "test_dataset": test_dataset_name,
                "n_samples": int(len(sample_metrics)),
                "dice_od_mean": float(sample_metrics["dice_od"].mean()),
                "dice_oc_mean": float(sample_metrics["dice_oc"].mean()),
                "mean_dice_od_oc": float(sample_metrics["mean_dice_od_oc"].mean()),
                "iou_od_mean": float(sample_metrics["iou_od"].mean()),
                "iou_oc_mean": float(sample_metrics["iou_oc"].mean()),
                "mean_iou_od_oc": float(sample_metrics["mean_iou_od_oc"].mean()),
                "centroid_error_od_px_mean": float(sample_metrics["centroid_error_od_px"].mean()),
                "centroid_error_oc_px_mean": float(sample_metrics["centroid_error_oc_px"].mean()),
                "area_error_od_rel_mean": float(sample_metrics["area_error_od_rel"].mean()),
                "area_error_oc_rel_mean": float(sample_metrics["area_error_oc_rel"].mean()),
                "pred_empty_od": int(sample_metrics["pred_empty_od"].sum()),
                "pred_empty_oc": int(sample_metrics["pred_empty_oc"].sum()),
                "checkpoint_path": sample_metrics["checkpoint_path"].iloc[0],
                "status": "loaded_existing",
            }
        else:
            sample_metrics, summary = evaluate_multisource_model_on_dataset(
                model_name=model_name,
                test_dataset_name=test_dataset_name,
            )

            sample_metrics.to_csv(sample_csv, index=False)
            summary["status"] = "evaluated"

        multisource_eval_rows.append(summary)

        print(
            f"{model_name} | train=MULTISOURCE | test={test_dataset_name} | "
            f"mean_dice={summary['mean_dice_od_oc']:.4f} | "
            f"OD={summary['dice_od_mean']:.4f} | "
            f"OC={summary['dice_oc_mean']:.4f} | "
            f"{summary['status']}"
        )

multisource_evaluation_summary = pd.DataFrame(multisource_eval_rows)

multisource_evaluation_summary_csv = (
    RESULTS_05_DIR / "notebook05_multisource_evaluation_summary.csv"
)

multisource_evaluation_summary.to_csv(multisource_evaluation_summary_csv, index=False)

multisource_model_summary = (
    multisource_evaluation_summary.groupby("model_name")
    .agg(
        n_test_domains=("test_dataset", "count"),
        mean_test_dice=("mean_dice_od_oc", "mean"),
        mean_test_dice_od=("dice_od_mean", "mean"),
        mean_test_dice_oc=("dice_oc_mean", "mean"),
        min_test_dice=("mean_dice_od_oc", "min"),
        max_test_dice=("mean_dice_od_oc", "max"),
        total_pred_empty_od=("pred_empty_od", "sum"),
        total_pred_empty_oc=("pred_empty_oc", "sum"),
    )
    .reset_index()
    .sort_values(["mean_test_dice", "min_test_dice"], ascending=False)
)

multisource_model_summary_csv = (
    RESULTS_05_DIR / "notebook05_multisource_model_summary.csv"
)

multisource_model_summary.to_csv(multisource_model_summary_csv, index=False)

print("\nMultisource evaluation summary:")
print(
    multisource_evaluation_summary[
        [
            "model_name",
            "test_dataset",
            "n_samples",
            "mean_dice_od_oc",
            "dice_od_mean",
            "dice_oc_mean",
            "pred_empty_od",
            "pred_empty_oc",
        ]
    ].to_string(index=False)
)

print("\nMultisource model summary:")
print(multisource_model_summary.to_string(index=False))

print(f"\nMultisource evaluation summary: {multisource_evaluation_summary_csv}")
print(f"Multisource model summary: {multisource_model_summary_csv}")

if len(multisource_evaluation_summary) != 6:
    raise RuntimeError(f"Expected 6 multisource evaluation rows, found {len(multisource_evaluation_summary)}")

print("Status: PASS")

In [ ]:
# Cell 27: Final segmentor selection from single-source and multisource evaluations

SINGLE_SOURCE_EVAL_CSV = RESULTS_05_DIR / "notebook05_cross_dataset_evaluation_summary.csv"
MULTISOURCE_EVAL_CSV = RESULTS_05_DIR / "notebook05_multisource_evaluation_summary.csv"
MULTISOURCE_TRAINING_CSV = RESULTS_05_DIR / "notebook05_multisource_training_results_consolidated.csv"

for path in [SINGLE_SOURCE_EVAL_CSV, MULTISOURCE_EVAL_CSV, MULTISOURCE_TRAINING_CSV]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

single_source_eval = pd.read_csv(SINGLE_SOURCE_EVAL_CSV)
multisource_eval = pd.read_csv(MULTISOURCE_EVAL_CSV)
multisource_training_results = pd.read_csv(MULTISOURCE_TRAINING_CSV)

single_source_eval = single_source_eval.copy()
single_source_eval["training_strategy"] = "single_source"

multisource_eval = multisource_eval.copy()
multisource_eval["training_strategy"] = "multisource"
multisource_eval["evaluation_type"] = "multisource_test"

comparison_columns = [
    "model_name",
    "training_strategy",
    "train_dataset",
    "test_dataset",
    "evaluation_type",
    "n_samples",
    "mean_dice_od_oc",
    "dice_od_mean",
    "dice_oc_mean",
    "mean_iou_od_oc",
    "pred_empty_od",
    "pred_empty_oc",
]

available_comparison_columns = [
    col for col in comparison_columns
    if col in single_source_eval.columns or col in multisource_eval.columns
]

all_evaluations = pd.concat(
    [
        single_source_eval.reindex(columns=available_comparison_columns),
        multisource_eval.reindex(columns=available_comparison_columns),
    ],
    axis=0,
).reset_index(drop=True)

strategy_summary = (
    all_evaluations.groupby(["training_strategy", "model_name"])
    .agg(
        n_pairs=("mean_dice_od_oc", "count"),
        mean_dice=("mean_dice_od_oc", "mean"),
        min_dice=("mean_dice_od_oc", "min"),
        max_dice=("mean_dice_od_oc", "max"),
        mean_dice_od=("dice_od_mean", "mean"),
        mean_dice_oc=("dice_oc_mean", "mean"),
        total_pred_empty_od=("pred_empty_od", "sum"),
        total_pred_empty_oc=("pred_empty_oc", "sum"),
    )
    .reset_index()
)

strategy_summary["total_empty_predictions"] = (
    strategy_summary["total_pred_empty_od"] +
    strategy_summary["total_pred_empty_oc"]
)

strategy_summary["selection_score"] = (
    strategy_summary["mean_dice"]
    + 0.25 * strategy_summary["min_dice"]
    - 0.001 * strategy_summary["total_empty_predictions"]
)

strategy_summary = strategy_summary.sort_values(
    ["selection_score", "mean_dice", "min_dice"],
    ascending=False,
).reset_index(drop=True)

multisource_domain_summary = (
    multisource_eval.pivot_table(
        index="model_name",
        columns="test_dataset",
        values="mean_dice_od_oc",
        aggfunc="mean",
    )
    .reset_index()
)

single_source_best_by_test = (
    single_source_eval.groupby(["model_name", "test_dataset"])
    .agg(
        best_single_source_test_dice=("mean_dice_od_oc", "max"),
        mean_single_source_test_dice=("mean_dice_od_oc", "mean"),
    )
    .reset_index()
)

multisource_vs_single = multisource_eval.merge(
    single_source_best_by_test,
    on=["model_name", "test_dataset"],
    how="left",
)

multisource_vs_single["delta_vs_best_single_source"] = (
    multisource_vs_single["mean_dice_od_oc"] -
    multisource_vs_single["best_single_source_test_dice"]
)

multisource_vs_single["delta_vs_mean_single_source"] = (
    multisource_vs_single["mean_dice_od_oc"] -
    multisource_vs_single["mean_single_source_test_dice"]
)

selected_row = strategy_summary.iloc[0]

FINAL_SEGMENTOR_MODEL = selected_row["model_name"]
FINAL_SEGMENTOR_STRATEGY = selected_row["training_strategy"]

if FINAL_SEGMENTOR_STRATEGY != "multisource":
    raise RuntimeError(
        f"Unexpected final strategy: {FINAL_SEGMENTOR_STRATEGY}. "
        "Final model should be selected from multisource candidates."
    )

final_checkpoint = (
    MODEL_ROOT /
    "multisource_final" /
    FINAL_SEGMENTOR_MODEL /
    "checkpoints" /
    "best_checkpoint.pt"
)

if not final_checkpoint.exists():
    raise FileNotFoundError(f"Missing final checkpoint: {final_checkpoint}")

final_training_row = multisource_training_results[
    multisource_training_results["model_name"] == FINAL_SEGMENTOR_MODEL
].iloc[0].to_dict()

final_segmentor_decision = {
    "final_segmentor_model": FINAL_SEGMENTOR_MODEL,
    "final_training_strategy": FINAL_SEGMENTOR_STRATEGY,
    "final_checkpoint": str(final_checkpoint),
    "selection_metric": "selection_score = mean_dice + 0.25*min_dice - 0.001*empty_predictions",
    "mean_test_dice": float(selected_row["mean_dice"]),
    "min_test_dice": float(selected_row["min_dice"]),
    "mean_test_dice_od": float(selected_row["mean_dice_od"]),
    "mean_test_dice_oc": float(selected_row["mean_dice_oc"]),
    "total_empty_predictions": int(selected_row["total_empty_predictions"]),
    "best_epoch_one_indexed": int(final_training_row["best_epoch_one_indexed"]),
    "training_best_val_mean_dice_od_oc": float(final_training_row["best_val_mean_dice_od_oc"]),
    "decision_rationale": (
        "ResidualUNetLite with multisource training is selected because it provides the "
        "best overall test Dice across REFUGE, PAPILA, and RIGA, retains strong OD/OC "
        "balance, and is substantially more computationally efficient than DeepLabLite."
    ),
}

all_evaluations_csv = RESULTS_05_DIR / "notebook05_all_segmentor_evaluations_combined.csv"
strategy_summary_csv = RESULTS_05_DIR / "notebook05_final_strategy_summary.csv"
multisource_domain_summary_csv = RESULTS_05_DIR / "notebook05_multisource_domain_summary.csv"
multisource_vs_single_csv = RESULTS_05_DIR / "notebook05_multisource_vs_single_source.csv"
final_decision_json = RESULTS_05_DIR / "notebook05_final_segmentor_decision.json"

all_evaluations.to_csv(all_evaluations_csv, index=False)
strategy_summary.to_csv(strategy_summary_csv, index=False)
multisource_domain_summary.to_csv(multisource_domain_summary_csv, index=False)
multisource_vs_single.to_csv(multisource_vs_single_csv, index=False)

with open(final_decision_json, "w", encoding="utf-8") as f:
    json.dump(final_segmentor_decision, f, indent=2)

print("Final strategy summary:")
print(strategy_summary.to_string(index=False))

print("\nMultisource domain summary:")
print(multisource_domain_summary.to_string(index=False))

print("\nMultisource vs single-source:")
print(
    multisource_vs_single[
        [
            "model_name",
            "test_dataset",
            "mean_dice_od_oc",
            "best_single_source_test_dice",
            "delta_vs_best_single_source",
            "delta_vs_mean_single_source",
        ]
    ].to_string(index=False)
)

print("\nFinal segmentor decision:")
print(json.dumps(final_segmentor_decision, indent=2))

print(f"\nFinal decision: {final_decision_json}")

if FINAL_SEGMENTOR_MODEL != "ResidualUNetLite":
    raise RuntimeError(f"Unexpected final model: {FINAL_SEGMENTOR_MODEL}")

print("Status: PASS")

In [ ]:
# Cell 28: Final status

from pathlib import Path
import json
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
RESULTS_05_DIR = JOURNAL_ROOT / "results" / "05_od_oc_segmentor_training"

SCREENING_MODELS_EXPECTED = [
    "CompactUNet",
    "ResidualUNetLite",
    "AttentionUNetLite",
    "DeepLabLite",
    "FPNLite",
]

FULL_TRAINING_MODELS_EXPECTED = [
    "DeepLabLite",
    "ResidualUNetLite",
]

MULTISOURCE_MODELS_EXPECTED = [
    "DeepLabLite",
    "ResidualUNetLite",
]

required_files = {
    "runtime_config": RESULTS_05_DIR / "notebook05_runtime_config.json",
    "dataloader_smoke_test": RESULTS_05_DIR / "notebook05_dataloader_smoke_test.csv",
    "model_registry_smoke_test": RESULTS_05_DIR / "notebook05_model_registry_smoke_test.csv",
    "architecture_screening_results": RESULTS_05_DIR / "notebook05_architecture_screening_results.csv",
    "selected_architectures": RESULTS_05_DIR / "notebook05_selected_architectures.csv",
    "full_training_results": RESULTS_05_DIR / "notebook05_full_training_results_consolidated.csv",
    "single_source_cross_dataset_evaluation": RESULTS_05_DIR / "notebook05_cross_dataset_evaluation_summary.csv",
    "architecture_decision_summary": RESULTS_05_DIR / "notebook05_architecture_decision_summary.csv",
    "multisource_training_results": RESULTS_05_DIR / "notebook05_multisource_training_results_consolidated.csv",
    "multisource_evaluation_summary": RESULTS_05_DIR / "notebook05_multisource_evaluation_summary.csv",
    "final_strategy_summary": RESULTS_05_DIR / "notebook05_final_strategy_summary.csv",
    "final_segmentor_decision": RESULTS_05_DIR / "notebook05_final_segmentor_decision.json",
}

file_checks = {
    key: path.exists()
    for key, path in required_files.items()
}

missing_required_files = [
    key for key, exists in file_checks.items()
    if not exists
]

if missing_required_files:
    raise FileNotFoundError(f"Missing required files: {missing_required_files}")

architecture_screening_results = pd.read_csv(required_files["architecture_screening_results"])
selected_architectures = pd.read_csv(required_files["selected_architectures"])
full_training_results = pd.read_csv(required_files["full_training_results"])
cross_dataset_evaluation_summary = pd.read_csv(required_files["single_source_cross_dataset_evaluation"])
architecture_decision_summary = pd.read_csv(required_files["architecture_decision_summary"])
multisource_training_results = pd.read_csv(required_files["multisource_training_results"])
multisource_evaluation_summary = pd.read_csv(required_files["multisource_evaluation_summary"])
final_strategy_summary = pd.read_csv(required_files["final_strategy_summary"])

with open(required_files["final_segmentor_decision"], "r", encoding="utf-8") as f:
    final_decision = json.load(f)

final_checkpoint_path = Path(final_decision["final_checkpoint"])

selected_full_models = selected_architectures.loc[
    selected_architectures["decision"] == "selected_for_full_training",
    "model_name",
].tolist()

content_checks = {
    "final_model_is_residual_unet_lite": final_decision["final_segmentor_model"] == "ResidualUNetLite",
    "final_strategy_is_multisource": final_decision["final_training_strategy"] == "multisource",
    "final_checkpoint_exists": final_checkpoint_path.exists(),
    "expected_screening_rows": len(architecture_screening_results) == 15,
    "expected_screening_models": sorted(architecture_screening_results["model_name"].unique().tolist()) == sorted(SCREENING_MODELS_EXPECTED),
    "expected_selected_full_training_models": sorted(selected_full_models) == sorted(FULL_TRAINING_MODELS_EXPECTED),
    "expected_full_training_runs": len(full_training_results) == 6,
    "expected_single_source_evaluations": len(cross_dataset_evaluation_summary) == 18,
    "expected_multisource_training_runs": len(multisource_training_results) == 2,
    "expected_multisource_evaluations": len(multisource_evaluation_summary) == 6,
    "expected_final_strategy_rows": len(final_strategy_summary) == 4,
}

final_checks = {**file_checks, **content_checks}
failed_checks = [key for key, value in final_checks.items() if not value]
status = "PASS" if not failed_checks else "FAIL"

notebook05_status = {
    "notebook": "05_train_external_od_oc_segmentors_cross_dataset_validation",
    "status": status,
    "screening_models": SCREENING_MODELS_EXPECTED,
    "selected_full_training_models": selected_full_models,
    "single_source_full_training_runs": int(len(full_training_results)),
    "single_source_evaluation_pairs": int(len(cross_dataset_evaluation_summary)),
    "multisource_models": MULTISOURCE_MODELS_EXPECTED,
    "multisource_training_runs": int(len(multisource_training_results)),
    "multisource_evaluation_pairs": int(len(multisource_evaluation_summary)),
    "final_segmentor_model": final_decision["final_segmentor_model"],
    "final_training_strategy": final_decision["final_training_strategy"],
    "final_checkpoint": final_decision["final_checkpoint"],
    "final_mean_test_dice": float(final_decision["mean_test_dice"]),
    "final_min_test_dice": float(final_decision["min_test_dice"]),
    "final_mean_test_dice_od": float(final_decision["mean_test_dice_od"]),
    "final_mean_test_dice_oc": float(final_decision["mean_test_dice_oc"]),
    "final_total_empty_predictions": int(final_decision["total_empty_predictions"]),
    "failed_checks": failed_checks,
    "next_notebook": "06_apply_final_od_oc_segmentor_to_hygd_anatomical_proxy.ipynb",
}

status_json = RESULTS_05_DIR / "notebook05_final_status.json"
status_txt = RESULTS_05_DIR / "notebook05_FINAL_STATUS.txt"
status_checks_csv = RESULTS_05_DIR / "notebook05_final_status_checks.csv"

with open(status_json, "w", encoding="utf-8") as f:
    json.dump(notebook05_status, f, indent=2)

with open(status_txt, "w", encoding="utf-8") as f:
    for key, value in notebook05_status.items():
        f.write(f"{key}: {value}\n")

pd.DataFrame(
    [{"check": key, "passed": bool(value)} for key, value in final_checks.items()]
).to_csv(status_checks_csv, index=False)

print(f"Status: {status}")
print(f"Final segmentor: {final_decision['final_segmentor_model']}")
print(f"Training strategy: {final_decision['final_training_strategy']}")
print(f"Final mean test Dice: {final_decision['mean_test_dice']:.6f}")
print(f"Final min test Dice: {final_decision['min_test_dice']:.6f}")
print(f"Final mean test Dice OD: {final_decision['mean_test_dice_od']:.6f}")
print(f"Final mean test Dice OC: {final_decision['mean_test_dice_oc']:.6f}")
print(f"Final total empty predictions: {final_decision['total_empty_predictions']}")
print(f"Final checkpoint: {final_decision['final_checkpoint']}")
print(f"Final status: {status_json}")

if failed_checks:
    raise RuntimeError(f"Notebook 05 failed checks: {failed_checks}")